<a href="https://colab.research.google.com/github/VasilisPapageorgiou/Bayesian-Embeddings/blob/main/Real_Data_bayesian_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# # Olist — manuscript-aligned Bayesian embedding budget allocation
# Binary target: estimated delivery interval > 24 days, one independent data split.
# MLP architecture: 32 -> 8 -> 1, ReLU, dropout=0,
# Adam(learning rate=0.001, weight_decay=0.0001), batch_size=32, 50 EPOCHS.
# NO EARLY STOPPING / NO PATIENCE / NO FIXED-DIMENSION PRIMARY EXPERIMENT.
# m is derived from the number of categorical predictors; K_j = min(N_j-1,p_z).
# Budget B means number of trainable embedding entries sum_j N_j*d_j.
# Same data-derived candidate budgets across ALL embedding allocation methods.
# Bayesian score E[g_j(d)] + xi SD[g_j(d)] is optimized by exact sparse DP.
# Select smallest B with inner-validation binary logloss <= Bmax logloss + zeta.
# Reference map is FROZEN after fitting and evaluates without dropout.
# Crucial methodological note: this is an explicitly documented BERNOULLI
# classification extension of the manuscript's Gaussian-response likelihood.
# Do not quote Gaussian-response theoretical guarantees for binary Y unchanged.
# 70/30 development/test and internal 80/20 train/budget validation; one split.
# The test split is NEVER used to choose budgets or epochs.
# GPU AUTO: CUDA/T4 with data-resident FP64 likelihood proposals, CPU fallback.
# USER UPLOAD: Colab prompts for all nine CSV files or one ZIP; NO Kaggle download.
# EXECUTION: EJOR and other non-Bayesian methods FIRST; Bayesian posterior and OUR refits LAST.
# PROGRESS: concise updates; full final tables/figures and checkpoint contents unchanged.
# STARTUP PROMPT: NEW study (archive old one) vs RESUME compatible checkpoints.
# SIM2-INSPIRED: full PT replica/RNG checkpoint, cold-chain diagnostics, swap roundtrips,
# cold-state cache audit, never promoting nonconverged draws to validated results.
# CPU-only NumPy/SciPy are retained for spectral proposals and knapsack DP.

import os, sys, glob, json, time, math, random, copy, shutil, hashlib, warnings, zipfile, re
from pathlib import Path
from dataclasses import dataclass, asdict
from datetime import datetime
from typing import Dict, List, Tuple
import numpy as np
import pandas as pd
from scipy.special import expit, log_expit, betaln, gammaln
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, roc_auc_score, average_precision_score, log_loss,
                             brier_score_loss, accuracy_score, f1_score,
                             precision_score, recall_score, balanced_accuracy_score,
                             matthews_corrcoef, roc_curve, precision_recall_curve)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm
try:
    import arviz as az
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable,'-m','pip','-q','install','arviz'])
    import arviz as az
warnings.filterwarnings('ignore',category=FutureWarning)

# ---------- All user-editable settings ----------
@dataclass
class Config:
    input_dir: str = ''
    output_dir: str = ''
    seed: int = 42
    outer_validation_fraction: float = 0.30  # strictly held-out TEST, never budget tuning
    inner_validation_fraction: float = 0.20
    batch_size: int = 32               # EmbDim.ipynb
    epochs: int = 50                   # fixed epochs for every candidate and final refit
    pilot_epochs: int = 50             # fixed epochs, no patience
    lr: float = 1e-3                   # EmbDim.ipynb
    weight_decay: float = 1e-4         # EmbDim.ipynb
    hidden_dims: tuple = (32, 8)       # EmbDim.ipynb MLP, including reference map
    dropout: float = 0.0               # Zero dropout for reference and all candidate MLPs
    # p_z is automatically the final hidden-layer width, not a fixed d_j or fixed m.
    budget_fractions: tuple = (0., .10, .22, .36, .50, .65, .80, 1.)
    budget_tolerance: float = 0.01     # Bernoulli log loss, as in EmbDim.ipynb
    posterior_xi: float = 1.0          # predeclared uncertainty penalty for OUR method
    run_exact_bayes: bool = True
    bayes_max_rows: int = 0            # 0=FULL inner-training likelihood, paper-aligned.
    # A positive subsample size is for pilots ONLY, not full-data posterior claims.
    pt_chains: int = 4
    pt_temps: int = 3
    pt_min_beta: float = 0.72
    pt_base: int = 3000
    pt_extend: int = 1500
    pt_max: int = 12000
    pt_burn: int = 900
    pt_thin: int = 5
    pt_hot_update_every: int = 3
    checkpoint_minutes: float = 20.0
    rhat_limit: float = 1.01
    ess_bulk_min: float = 400.
    ess_tail_min: float = 400.
    require_bayes_convergence: bool = True
    bootstrap_reps: int = 0
    plot_dpi: int = 220
    smoke_test: bool = False          # must be False for the actual study
    gpu_posterior_likelihood: bool = True  # CUDA float64 frozen-map LL, CPU NumPy fallback

CFG=Config()
# Presentation-only controls: roughly half as many progress messages.
# These do not affect model training, Bayesian inference, saved diagnostics or results.
PROGRESS_INTERVAL_SECONDS = 900.0  # No frequent likelihood/iteration heartbeats
EPOCH_REPORT_EVERY = 50            # One training summary per 50-epoch fit
MCMC_ITER_REPORT_EVERY = 1000      # Log MCMC at most every 1,000 iterations
CHECKPOINT_REPORT_EVERY = 50       # Show only final training checkpoint
TQDM_MIN_INTERVAL_SECONDS = 30.0   # Tqdm rendering is disabled below
# Dynamic working dimension is derived from the common MLP architecture.
# It is NOT an allocated predictor dimension and does not set the number of predictors.
REFERENCE_PZ = int(CFG.hidden_dims[-1])
if CFG.smoke_test:
    CFG.epochs=2;CFG.pilot_epochs=2;CFG.pt_chains=2;CFG.pt_temps=3
    CFG.pt_base=20;CFG.pt_max=20;CFG.pt_burn=5;CFG.pt_thin=2
    CFG.bayes_max_rows=400;CFG.bootstrap_reps=0
    CFG.budget_fractions=(0.,.5,1.)
    CFG.require_bayes_convergence=False
    CFG.output_dir=CFG.output_dir or 'OLIST_BUDGET_ALIGNED_SMOKE_TEST'
# Prefer a Colab T4 (or any available CUDA GPU); otherwise use CPU automatically.
# CUDA is probed so a visible but unusable GPU does not break the experiment.
def select_device():
    if torch.cuda.is_available():
        try:
            candidate=torch.device('cuda:0')
            probe=torch.ones(8,device=candidate,dtype=torch.float64)
            torch.cuda.synchronize(candidate)
            if float(probe.sum().item())!=8.:
                raise RuntimeError('CUDA probe returned an incorrect value')
            name=torch.cuda.get_device_name(0)
            free,total=torch.cuda.mem_get_info(0)
            print(f'GPU enabled: {name} | available VRAM {free/2**30:.2f}/{total/2**30:.2f} GiB',flush=True)
            return candidate
        except Exception as exc:
            print(f'GPU unavailable or failed validation ({exc!r}); using CPU.',flush=True)
    else:
        print('CUDA GPU not detected; using CPU.',flush=True)
    return torch.device('cpu')

DEVICE=select_device()
USE_CUDA=DEVICE.type=='cuda'
NTHREADS=max(1,min(os.cpu_count() or 1,4));torch.set_num_threads(NTHREADS)

def seed_everything(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
seed_everything(CFG.seed)

# Google Colab: mount Drive so checkpoints survive runtime disconnections.
try:
    from google.colab import drive as _colab_drive
    if not Path('/content/drive/MyDrive').exists():_colab_drive.mount('/content/drive')
except ImportError:
    pass

if CFG.output_dir:
    OUT=Path(CFG.output_dir).expanduser().resolve()
elif Path('/content/drive/MyDrive').exists():
    OUT=Path('/content/drive/MyDrive/JRSSB_Embedding_Allocation/OLIST_BAYESIAN_EJOR_GPU_MANUAL_UPLOAD_V4')
else:
    OUT=Path('OLIST_BAYESIAN_EJOR_GPU_MANUAL_UPLOAD_V4').resolve()

# The choice is made BEFORE any results/config files are written. This is
# deliberately similar to the explicit NEW/RESUME guard in Simulation 2 V11.
def choose_run_mode(out_dir):
    existing=out_dir.is_dir() and any(out_dir.iterdir())
    checkpoints=(out_dir/'reference_model_weights.pt',
                 out_dir/'posterior'/'posterior_checkpoint.pt',
                 out_dir/'tables'/'classical_budget_comparison.csv',
                 out_dir/'tables'/'classical_heldout_method_comparison.csv',
                 out_dir/'tables'/'heldout_method_comparison.csv')
    can_resume=any(p.exists() for p in checkpoints)
    print('\n========== OLIST EXPERIMENT: NEW OR RESUME ==========',flush=True)
    print('[1] START NEW — archive any prior results and reset training + MCMC',flush=True)
    print('[2] RESUME — reuse finished fits and exact Bayesian PT checkpoint',flush=True)
    print('Experiment directory:',out_dir,flush=True)
    if existing:
        print('Existing study detected; compatible saved work:',
              ('YES' if can_resume else 'NO'),flush=True)
    chosen=os.environ.get('OLIST_RUN_MODE','').strip().lower()
    choices={'1':'new','2':'resume','new':'new','resume':'resume'}
    while True:
        if chosen not in choices:
            chosen=input('Choose 1=NEW or 2=RESUME: ').strip().lower()
        mode=choices[chosen]
        if mode=='resume' and not can_resume:
            print('Cannot resume: no saved training/checkpoint found in this folder.',flush=True)
            chosen=''
            continue
        break
    archived=None
    if mode=='new' and existing:
        print('PREVIOUS results will be MOVED to a timestamped archive, not deleted.',flush=True)
        # As in SIM2, require explicit typed confirmation if executed interactively.
        if os.environ.get('OLIST_RUN_MODE','').strip()=='':
            confirmation=input('Type ARCHIVE to start NEW and preserve the old study: ').strip()
            if confirmation!='ARCHIVE':
                raise RuntimeError('Existing work left unchanged. Re-run and choose RESUME.')
        date=datetime.now().strftime('%Y%m%d_%H%M%S')
        archived=out_dir.with_name(out_dir.name+'_ARCHIVED_'+date)
        counter=1
        while archived.exists():
            archived=out_dir.with_name(out_dir.name+'_ARCHIVED_'+date+'_'+str(counter))
            counter+=1
        out_dir.rename(archived)
        out_dir.mkdir(parents=True,exist_ok=True)
        # Preserve access to explicitly uploaded data; do not download from Kaggle.
        old_uploads=archived/'user_supplied_olist_csvs'
        if old_uploads.is_dir():
            shutil.copytree(old_uploads,out_dir/'user_supplied_olist_csvs')
            print('Reused your uploaded CSVs in the NEW run.',flush=True)
        print('Archived prior study at:',archived,flush=True)
    elif mode=='new':
        out_dir.mkdir(parents=True,exist_ok=True)
    else:
        # Verify important scientific settings before opening any saved work.
        saved_config=out_dir/'configuration.json'
        if saved_config.exists():
            before=json.loads(saved_config.read_text())
            now=asdict(CFG)
            safe_changes={'checkpoint_minutes','pt_base','pt_extend','pt_max',
                          'bootstrap_reps','plot_dpi','require_bayes_convergence',
                          'input_dir','output_dir'}
            changed=[name for name in before if name in now and name not in safe_changes
                     and before[name] != (list(now[name]) if isinstance(now[name],tuple) else now[name])]
            if changed:
                raise RuntimeError('RESUME settings incompatible: '+', '.join(changed)+
                                   '. Use original settings or START NEW.')
        print('RESUMING prior study. Existing fits and posterior will not be erased.',flush=True)
    return mode,str(archived) if archived is not None else None

RUN_MODE,ARCHIVED_DIR=choose_run_mode(OUT)
FIG=OUT/'figures';TAB=OUT/'tables';POST=OUT/'posterior';PRED=OUT/'predictions';FITCACHE=OUT/'inner_fits'
MLP_CHECKPOINTS=OUT/'mlp_epoch_checkpoints'
MLP_CHECKPOINT_EVERY_EPOCHS=5  # periodic checkpoint, NOT patience/early stopping
for d in (OUT,FIG,TAB,POST,PRED,FITCACHE,MLP_CHECKPOINTS):d.mkdir(parents=True,exist_ok=True)
(OUT/'run_selection.json').write_text(json.dumps({
    'selected_mode':RUN_MODE,'archived_previous':ARCHIVED_DIR,
    'timestamp':datetime.now().isoformat(),'program':'Sim2-inspired resume/progress'},indent=2))
with open(OUT/'configuration.json','w') as f:json.dump(asdict(CFG),f,indent=2)
print('Compute device:',DEVICE,'| posterior LL:',
      ('CUDA float64' if USE_CUDA and CFG.gpu_posterior_likelihood else 'NumPy CPU'),
      '| output:',OUT,'| mode:',RUN_MODE,flush=True)

# Persistent human-readable progress visible both in the cell and in Google Drive.
PROGRESS_LOG=OUT/'execution_progress.log'
PROGRESS_STATE=OUT/'execution_progress.json'
def report(stage,message,completed=None,total=None):
    now=time.strftime('%Y-%m-%d %H:%M:%S')
    fraction=(f' [{completed}/{total}]' if completed is not None and total is not None else '')
    line=f'[{now}] {stage}{fraction}: {message}'
    print(line,flush=True)
    with PROGRESS_LOG.open('a',encoding='utf-8') as stream:
        stream.write(line+'\n');stream.flush()
    tmp=PROGRESS_STATE.with_suffix('.tmp')
    tmp.write_text(json.dumps({'time':now,'stage':stage,'completed':completed,'total':total,'message':message},indent=2))
    os.replace(tmp,PROGRESS_STATE)


# ---------- EXPLICIT USER UPLOAD (NO KaggleHub / NO automatic file search) ----------
# On the first Colab run, a browser file-picker asks for the nine original CSVs
# (all at once) OR a ZIP archive that contains them. Uploaded data persist under
# the output directory on Google Drive so resumed runs do not require re-upload.
FILES=['olist_orders_dataset','olist_order_items_dataset','olist_order_payments_dataset',
       'olist_customers_dataset','olist_products_dataset','olist_sellers_dataset',
       'olist_order_reviews_dataset','olist_geolocation_dataset',
       'product_category_name_translation']
USER_DATA=OUT/'user_supplied_olist_csvs'
USER_DATA.mkdir(parents=True,exist_ok=True)

def _dataset_key(filename):
    # Accept original Kaggle names and notebook-upload variants, e.g. (2).csv.
    name=Path(str(filename).replace('\\','/')).name
    for key in FILES:
        if re.fullmatch(re.escape(key)+r'(?:\s*\(\d+\))?\.csv',name,flags=re.I):
            return key
    return None

def _dataset_available(folder):
    return {k:str(folder/(k+'.csv')) for k in FILES
            if (folder/(k+'.csv')).is_file() and (folder/(k+'.csv')).stat().st_size>0}

def _store_user_selection(uploaded):
    # uploaded: {filename: file_bytes}, as returned by google.colab.files.upload().
    saved=[]
    for filename,blob in uploaded.items():
        if filename.lower().endswith('.zip'):
            import io
            with zipfile.ZipFile(io.BytesIO(blob)) as archive:
                for member in archive.infolist():
                    if member.is_dir():continue
                    key=_dataset_key(member.filename)
                    if key is None:continue
                    if member.file_size==0:raise ValueError('Empty CSV in ZIP: '+member.filename)
                    dest=USER_DATA/(key+'.csv')
                    with archive.open(member) as source,open(dest,'wb') as target:
                        shutil.copyfileobj(source,target,1024*1024)
                    saved.append(key)
        else:
            key=_dataset_key(filename)
            if key is None:
                print('Ignored (not an expected Olist CSV):',filename,flush=True)
                continue
            if not blob:raise ValueError('Empty uploaded CSV: '+filename)
            (USER_DATA/(key+'.csv')).write_bytes(blob)
            saved.append(key)
    return saved

def data_sources():
    # CFG.input_dir is an EXPLICIT choice by the user, not automatic discovery.
    if CFG.input_dir:
        folder=Path(CFG.input_dir).expanduser()
        if not folder.is_dir():raise FileNotFoundError('User-specified input folder does not exist: '+str(folder))
        candidates={}
        for f in folder.iterdir():
            if not f.is_file() or f.stat().st_size==0:continue
            key=_dataset_key(f.name)
            if key is not None:candidates.setdefault(key,str(f))
        missing=[k for k in FILES if k not in candidates]
        if missing:raise FileNotFoundError('Missing required CSVs in CFG.input_dir: '+', '.join(missing))
        print('Using files from the folder you explicitly specified:',folder,flush=True)
        return {k:candidates[k] for k in FILES}

    present=_dataset_available(USER_DATA)
    if len(present)!=len(FILES):
        # There is deliberately NO Kaggle download and NO silent scan of /mnt/data.
        try:
            from google.colab import files as colab_files
        except ImportError:
            print('Please provide the Olist dataset. Enter the path to a folder containing all nine CSV files.')
            print('Required files:',', '.join(k+'.csv' for k in FILES))
            folder=Path(input('Olist CSV folder: ').strip().strip('\"')).expanduser()
            if not folder.is_dir():raise FileNotFoundError('The supplied folder does not exist: '+str(folder))
            candidates={}
            for f in folder.iterdir():
                key=_dataset_key(f.name) if f.is_file() else None
                if key and f.stat().st_size:candidates.setdefault(key,str(f))
            missing=[k for k in FILES if k not in candidates]
            if missing:raise FileNotFoundError('Files missing from supplied folder: '+', '.join(missing))
            return {k:candidates[k] for k in FILES}
        while len(present)!=len(FILES):
            missing=[k+'.csv' for k in FILES if k not in present]
            print('\nACTION REQUIRED: upload your Olist dataset through the Colab file picker.',flush=True)
            print('Choose all nine CSV files in one selection, OR upload one ZIP containing them.',flush=True)
            print('Files still needed:',', '.join(missing),flush=True)
            uploaded=colab_files.upload()  # Opens the browser's file selection dialogue.
            if not uploaded:
                raise RuntimeError('No dataset files were uploaded. Run the cell again and select the files.')
            saved=_store_user_selection(uploaded)
            uploaded.clear()  # Release uploaded byte buffers (geolocation CSV is large).
            del uploaded
            if not saved:
                raise ValueError('None of the selected files matches the required Olist filenames.')
            print('Accepted:',', '.join(sorted(set(saved))),flush=True)
            present=_dataset_available(USER_DATA)
    else:
        print('Reusing the Olist files that YOU uploaded previously, saved at:',USER_DATA,flush=True)
    print('All nine user-supplied Olist CSVs verified. No Kaggle download performed.',flush=True)
    manifest={k:{'path':present[k],'bytes':Path(present[k]).stat().st_size} for k in FILES}
    (OUT/'user_dataset_manifest.json').write_text(json.dumps(manifest,indent=2))
    return {k:present[k] for k in FILES}

def read_table(paths,name,columns=None):
    return pd.read_csv(paths[name],usecols=columns)

# ---------- One row per order; unavailable/post-outcome fields NEVER predictors ----------
# Exclude the high-cardinality seller identifier from all embedding methods.
# Seller state and numerical seller_count remain available as predictors.
CATS=['primary_product_category','customer_state','seller_state',
      'primary_payment_type','purchase_month','purchase_weekday']
NUMS=['item_count','unique_products','seller_count','total_item_price','total_freight_value',
      'mean_item_price','max_item_price','mean_product_weight_g',
      'mean_product_length_cm','mean_product_height_cm','mean_product_width_cm',
      'mean_product_description_length','mean_product_photos_qty',
      'payment_total','payment_count','max_payment_installments']
TARGET='estimated_duration_over_24d'

def build_olist(paths):
    order=read_table(paths,'olist_orders_dataset')
    cust=read_table(paths,'olist_customers_dataset', ['customer_id','customer_state'])
    items=read_table(paths,'olist_order_items_dataset',
        ['order_id','order_item_id','product_id','seller_id','price','freight_value'])
    products=read_table(paths,'olist_products_dataset',
        ['product_id','product_category_name','product_description_lenght','product_photos_qty',
         'product_weight_g','product_length_cm','product_height_cm','product_width_cm'])
    sellers=read_table(paths,'olist_sellers_dataset',['seller_id','seller_state'])
    pay=read_table(paths,'olist_order_payments_dataset',
        ['order_id','payment_sequential','payment_type','payment_installments','payment_value'])
    if order.order_id.duplicated().any():raise AssertionError('orders are not unique')
    if products.product_id.duplicated().any():raise AssertionError('products are not unique')
    if sellers.seller_id.duplicated().any():raise AssertionError('sellers are not unique')
    order['purchase_dt']=pd.to_datetime(order['order_purchase_timestamp'],errors='coerce')
    order['estimated_dt']=pd.to_datetime(order['order_estimated_delivery_date'],errors='coerce')
    order['estimated_delivery_days']=(order.estimated_dt-order.purchase_dt).dt.total_seconds()/86400.
    # Strict interpretation of >24 days, not rounded inclusive day counts.
    order=order[order['estimated_delivery_days'].notna() &
                order['purchase_dt'].notna()].copy()
    order[TARGET]=(order['estimated_delivery_days']>24.).astype(np.int8)
    order['purchase_month']=order.purchase_dt.dt.month.astype(str)
    order['purchase_weekday']=order.purchase_dt.dt.dayofweek.astype(str)
    items=items.merge(products,on='product_id',how='left',validate='many_to_one')
    items=items.merge(sellers,on='seller_id',how='left',validate='many_to_one')
    # Deterministic primary product and seller state, without seller ID as a predictor.
    prim=(items.sort_values(['order_id','price','order_item_id'],
                             ascending=[True,False,True],kind='mergesort')
               .drop_duplicates('order_id')[['order_id','seller_state','product_category_name']]
               .rename(columns={'product_category_name':'primary_product_category'}))
    g=items.groupby('order_id',sort=False)
    agg=g.agg(item_count=('order_item_id','size'),unique_products=('product_id','nunique'),
              seller_count=('seller_id','nunique'),total_item_price=('price','sum'),
              total_freight_value=('freight_value','sum'),mean_item_price=('price','mean'),
              max_item_price=('price','max'),mean_product_weight_g=('product_weight_g','mean'),
              mean_product_length_cm=('product_length_cm','mean'),
              mean_product_height_cm=('product_height_cm','mean'),
              mean_product_width_cm=('product_width_cm','mean'),
              mean_product_description_length=('product_description_lenght','mean'),
              mean_product_photos_qty=('product_photos_qty','mean')).reset_index()
    payagg=pay.groupby('order_id',sort=False).agg(payment_total=('payment_value','sum'),
                   payment_count=('payment_sequential','size'),
                   max_payment_installments=('payment_installments','max')).reset_index()
    paytype=(pay.sort_values(['order_id','payment_value','payment_sequential'],
                             ascending=[True,False,True],kind='mergesort')
             .drop_duplicates('order_id')[['order_id','payment_type']]
             .rename(columns={'payment_type':'primary_payment_type'}))
    df=(order[['order_id','customer_id',TARGET,'purchase_month','purchase_weekday']]
        .merge(cust,on='customer_id',how='left',validate='many_to_one')
        .merge(agg,on='order_id',how='left',validate='one_to_one')
        .merge(prim,on='order_id',how='left',validate='one_to_one')
        .merge(payagg,on='order_id',how='left',validate='one_to_one')
        .merge(paytype,on='order_id',how='left',validate='one_to_one'))
    assert df.order_id.is_unique and len(df)==len(order)
    assert not any(x in CATS+NUMS for x in ('estimated_delivery_days','order_estimated_delivery_date',
            'order_delivered_customer_date','order_delivered_carrier_date','shipping_limit_date',
            'seller_geolat','seller_geolng','shipping_distance_km','order_status'))
    return df[['order_id',TARGET]+CATS+NUMS].copy()

# ---------- Locked inner-fitted categorical vocabulary/numeric scalers ----------
class Prep:
    def fit(self, df):
        self.vocabs={};self.levels=[]
        for col in CATS:
            vals=df[col].fillna('__MISSING__').astype(str)
            cats=sorted(vals.unique());self.vocabs[col]={v:i+1 for i,v in enumerate(cats)}
            self.levels.append(len(cats))  # 0 is UNKNOWN, no trainable emb vector
            weights=vals.value_counts().reindex(cats,fill_value=0).to_numpy(dtype=float)
            if not hasattr(self,'pis'):self.pis=[]
            self.pis.append(weights/weights.sum())
        self.imputer=SimpleImputer(strategy='median',keep_empty_features=True)
        self.scaler=StandardScaler()
        X=self.imputer.fit_transform(df[NUMS].apply(pd.to_numeric,errors='coerce'))
        self.scaler.fit(X)
        return self
    def transform(self,df):
        C=np.stack([df[k].fillna('__MISSING__').astype(str).map(self.vocabs[k]).fillna(0).astype('int64').values
                     for k in CATS],axis=1)
        nums=self.imputer.transform(df[NUMS].apply(pd.to_numeric,errors='coerce'))
        X=self.scaler.transform(nums).astype('float32')
        return C,X,df[TARGET].values.astype('float32')

def _mlp_layers(input_size):
    # Exact feedforward architecture of EmbDim.ipynb, with a logit output.
    layers=[];prev=int(input_size)
    for width in CFG.hidden_dims:
        layers.extend([nn.Linear(prev,int(width)),nn.ReLU(),nn.Dropout(CFG.dropout)])
        prev=int(width)
    layers.append(nn.Linear(prev,1))
    return nn.Sequential(*layers)

class EmbeddingMLP(nn.Module):
    def __init__(self,levels,dims,n_num=16):
        super().__init__()
        if not (len(levels)==len(dims) and all(1<=d<n for d,n in zip(dims,levels))):
            raise ValueError('Need one dimension per categorical predictor, strictly < categories')
        self.dims=tuple(int(d) for d in dims)
        self.emb=nn.ModuleList([nn.Embedding(n+1,d,padding_idx=0) for n,d in zip(levels,self.dims)])
        self.net=_mlp_layers(sum(self.dims)+n_num)
    def forward(self,cat,num):
        z=torch.cat([self.emb[j](cat[:,j]) for j in range(len(self.emb))]+[num],dim=1)
        return self.net(z).squeeze(-1)

class OneHotMLP(nn.Module):
    # Algebraically equivalent to a full one-hot input + EmbDim MLP first layer,
    # but avoids materializing a huge dense one-hot array.
    def __init__(self,levels,n_num=16):
        super().__init__()
        width=int(CFG.hidden_dims[0])
        self.cat_first=nn.ModuleList([nn.Embedding(n+1,width,padding_idx=0) for n in levels])
        self.num_first=nn.Linear(n_num,width)
        layers=[nn.ReLU(),nn.Dropout(CFG.dropout)]
        prev=width
        for h in CFG.hidden_dims[1:]:
            layers.extend([nn.Linear(prev,int(h)),nn.ReLU(),nn.Dropout(CFG.dropout)])
            prev=int(h)
        layers.append(nn.Linear(prev,1))
        self.rest=nn.Sequential(*layers)
    def forward(self,cat,num):
        h=self.num_first(num)
        for j,e in enumerate(self.cat_first):h=h+e(cat[:,j])
        return self.rest(h).squeeze(-1)

class NumericMLP(nn.Module):
    def __init__(self,n_num=16):
        super().__init__();self.net=_mlp_layers(n_num)
    def forward(self,cat,num):return self.net(num).squeeze(-1)

class ReferenceMLP(nn.Module):
    # Paper's additive rich common latent reference, rather than concatenated reduced model.
    # Phi uses the SAME hidden layers/activations/dropout as all comparisons.
    def __init__(self,levels,n_num,pis):
        super().__init__();p=REFERENCE_PZ
        self.emb=nn.ModuleList([nn.Embedding(n+1,p,padding_idx=0) for n in levels])
        self.gamma=nn.Linear(n_num,p)
        self.phi=_mlp_layers(p)
        for j,pi in enumerate(pis):
            self.register_buffer('pi_'+str(j),torch.as_tensor(pi,dtype=torch.float32))
    def forward(self,cat,num):
        z=self.gamma(num)
        for j,e in enumerate(self.emb):z=z+e(cat[:,j])
        return self.phi(z).squeeze(-1)
    @torch.no_grad()
    def center(self):
        # Weighted centering pi_j^T Theta_j = 0, as in the manuscript.
        # The additive shift is transferred to the latent intercept exactly.
        for j,e in enumerate(self.emb):
            pi=getattr(self,'pi_'+str(j))
            shift=(pi[:,None]*e.weight[1:]).sum(dim=0,keepdim=True)
            e.weight[1:]-=shift
            self.gamma.bias+=shift.squeeze(0)
            e.weight[0].zero_()

def count_parameters(model):
    # Exclude the frozen UNKNOWN padding row from embedding-parameter counts.
    total=sum(int(p.numel()) for p in model.parameters() if p.requires_grad)
    for module in model.modules():
        if isinstance(module,nn.Embedding) and module.padding_idx is not None:
            total-=int(module.embedding_dim)
    return total
def embedding_cost(levels,dims):return int(sum(int(ni)*int(di) for ni,di in zip(levels,dims)))

def loader(data,shuffle=False,seed=42):
    c,x,y=data
    ds=TensorDataset(torch.as_tensor(c,dtype=torch.long),torch.as_tensor(x,dtype=torch.float32),
                     torch.as_tensor(y,dtype=torch.float32))
    gen=torch.Generator().manual_seed(seed)
    # Pinned host memory and non-blocking transfers speed up CUDA training.
    return DataLoader(ds,batch_size=CFG.batch_size,shuffle=shuffle,generator=gen,
                      pin_memory=USE_CUDA)

@torch.no_grad()
def probabilities(model, data):
    model.to(DEVICE).eval();c,x,y=data;res=[]
    for cb,xb,yb in loader(data,False):
        out=model(cb.to(DEVICE,non_blocking=USE_CUDA),xb.to(DEVICE,non_blocking=USE_CUDA));res.append(torch.sigmoid(out).cpu().numpy())
    return np.clip(np.concatenate(res),1e-7,1-1e-7)


def train_model(model, tr, valid=None, epochs=None, seed=42, progress_label=None):
    # Fixed-epoch optimizer, independent of validation performance. An MLP epoch
    # checkpoint contains the weights, Adam moments, epoch and RNG states. As in
    # SIM2, incomplete transitions are NEVER checkpointed. One cannot resume a
    # half-finished epoch, but up to (checkpoint_interval-1) epochs may be lost.
    seed_everything(seed)
    model.to(DEVICE)
    opt=torch.optim.Adam(model.parameters(),lr=CFG.lr,weight_decay=CFG.weight_decay)
    hist=[]
    epochs=CFG.epochs if epochs is None else int(epochs)
    steps_per_epoch=(len(tr[2])+CFG.batch_size-1)//CFG.batch_size
    label=f'{progress_label or type(model).__name__} | {len(tr[2]):,} rows | {epochs} epochs | {DEVICE.type}'
    signature=hashlib.sha256()
    signature.update(json.dumps(dict(label=progress_label or type(model).__name__,
        model=type(model).__name__, shapes=[(key,list(value.shape)) for key,value in model.state_dict().items()],
        batch_size=CFG.batch_size,epochs=epochs,lr=CFG.lr,weight_decay=CFG.weight_decay,
        hidden_dims=CFG.hidden_dims,dropout=CFG.dropout,seed=seed,
        n_valid=0 if valid is None else len(valid[2])),sort_keys=True).encode())
    for part in tr:
        signature.update(np.ascontiguousarray(part).tobytes())
    resume_signature=signature.hexdigest()
    resume_path=MLP_CHECKPOINTS/(resume_signature[:24]+'.pt')
    completed_epoch=0
    if RUN_MODE=='resume' and resume_path.exists():
        saved=torch.load(resume_path,map_location='cpu',weights_only=False)
        if saved.get('signature')!=resume_signature:
            raise RuntimeError('An MLP epoch checkpoint has an incompatible signature')
        completed_epoch=int(saved['completed_epoch'])
        if completed_epoch>epochs:
            raise RuntimeError('Saved epoch count exceeds the requested total')
        model.load_state_dict(saved['model'])
        opt.load_state_dict(saved['optimizer'])
        hist=list(saved['history'])
        if len(hist)!=completed_epoch:
            raise RuntimeError('MLP epoch checkpoint contains inconsistent history')
        torch.set_rng_state(saved['torch_cpu_rng'])
        if USE_CUDA and saved.get('torch_cuda_rng') is not None:
            try:torch.cuda.set_rng_state_all(saved['torch_cuda_rng'])
            except Exception as exc:print('WARNING: CUDA RNG restore unavailable:',exc,flush=True)
        np.random.set_state(saved['numpy_rng'])
        random.setstate(saved['python_rng'])
        print(f'  [MLP RESUME] {label}: restored weights, Adam state, RNG and epoch '
              f'{completed_epoch}/{epochs}',flush=True)
    elif RUN_MODE=='resume':
        print(f'  [MLP CHECKPOINT] no partial epoch state for {label}; begin at epoch 1',flush=True)

    print('START TRAINING:',label,'| saved epoch=',completed_epoch,flush=True)
    start_clock=time.monotonic()
    with tqdm(total=epochs*steps_per_epoch,initial=completed_epoch*steps_per_epoch,
              desc=label,unit='batch',mininterval=TQDM_MIN_INTERVAL_SECONDS,leave=False,dynamic_ncols=True,disable=True) as progress:
        for epoch in range(completed_epoch+1,epochs+1):
            model.train();losses=[]
            for batch_idx,(cb,xb,yb) in enumerate(loader(tr,True,seed+epoch),start=1):
                cb,xb,yb=(v.to(DEVICE,non_blocking=USE_CUDA) for v in (cb,xb,yb))
                opt.zero_grad(set_to_none=True)
                loss=F.binary_cross_entropy_with_logits(model(cb,xb),yb)
                loss.backward();opt.step()
                if isinstance(model,ReferenceMLP):model.center()
                losses.append((float(loss.item()),len(yb)))
                progress.update(1)
            train_loss=sum(v*n for v,n in losses)/sum(n for _,n in losses)
            vv=float(log_loss(valid[2],probabilities(model,valid))) if valid is not None else np.nan
            hist.append({'epoch':epoch,'train_logloss':train_loss,'inner_val_logloss':vv})
            progress.set_postfix(epoch=f'{epoch}/{epochs}',loss=f'{train_loss:.5f}',
                                 **({'val':f'{vv:.5f}'} if valid is not None else {}),refresh=True)
            val_info=f' | val_logloss={vv:.5f}' if valid is not None else ''
            if epoch % EPOCH_REPORT_EVERY == 0 or epoch == epochs:
                print(f'  {type(model).__name__}: epoch {epoch:>2}/{epochs}'
                      f' | train_logloss={train_loss:.5f}{val_info}'
                      f' | elapsed={((time.monotonic()-start_clock)/60):.1f} min',flush=True)
            if epoch%MLP_CHECKPOINT_EVERY_EPOCHS==0 or epoch==epochs:
                contents=dict(signature=resume_signature,completed_epoch=epoch,
                    model=model.state_dict(),optimizer=opt.state_dict(),history=hist,
                    torch_cpu_rng=torch.get_rng_state(),
                    torch_cuda_rng=(torch.cuda.get_rng_state_all() if USE_CUDA else None),
                    numpy_rng=np.random.get_state(),python_rng=random.getstate())
                tmp=resume_path.with_suffix('.tmp')
                torch.save(contents,tmp)
                os.replace(tmp,resume_path)
    return model,pd.DataFrame(hist),epochs

# ---------- Spectral approximation, continuous EJOR and budget strategies ----------
def spectral_pilot(model, levels):
    eigs=[];a=[];energy=[]
    for j,e in enumerate(model.emb):
        w=e.weight.detach().cpu().numpy()[1:]
        w=w-w.mean(0,keepdims=True)
        s=np.linalg.svd(w,compute_uv=False)
        eig=np.maximum(s[:min(REFERENCE_PZ,levels[j]-1)]**2,0)
        eigs.append(eig);energy.append(float(eig.sum()))
        # Least-squares a_j / d approximation to observed residual squared tails.
        # Fit all 1 <= d < K_j; no outcome use beyond TRAIN-fitted pilot.
        d=np.arange(1,len(eig),dtype=float)
        if len(d):
            residual=np.array([eig[int(k):].sum() for k in d]);u=1./d
            a_j=float(max(1e-12,np.dot(u,residual)/np.dot(u,u)))
        else:a_j=1e-12
        a.append(a_j)
    return eigs,np.asarray(a),np.asarray(energy)

def limits(levels):return np.asarray([min(REFERENCE_PZ,n-1) for n in levels],int)
def budget_grid(levels,K):
    lo=int(np.sum(levels));hi=embedding_cost(levels,K)
    result=sorted(set(int(lo+round((hi-lo)*x)) for x in CFG.budget_fractions))
    return [b for b in result if b>=lo]

def allocation_greedy(weights,levels,K,B,objective):
    # All baselines use the SAME cost convention N_j*d_j and d_j<K_j categories.
    n=np.asarray(levels,dtype=int);K=np.asarray(K,dtype=int)
    dims=np.ones(len(n),dtype=int);remain=B-int(np.dot(n,dims))
    if remain<0:raise ValueError('Infeasible budget')
    while True:
        can=np.where((dims<K)&(n<=remain))[0]
        if len(can)==0:break
        # Uniform: prefer smallest current dimension; cardinality: sqrt(N), etc.
        score=np.asarray([weights[j]/((dims[j]+.0001)**2*n[j]) for j in can])
        j=can[int(np.argmax(score))];dims[j]+=1;remain-=n[j]
    return tuple(int(v) for v in dims)

def allocate_ejor(a,n,K,B):
    # Continuous EJOR d_j = B sqrt(a_j/N_j) / sum_k sqrt(a_k N_k).
    # Integer projection uses marginal a_j/(d(d+1))/N_j to respect budget.
    n=np.asarray(n);a=np.maximum(a,1e-12);K=np.asarray(K)
    dcont=B*np.sqrt(a/n)/np.sum(np.sqrt(a*n))
    d=np.minimum(K,np.maximum(1,np.floor(dcont).astype(int)))
    while int(np.dot(n,d))>B:
        idx=np.where(d>1)[0]
        j=int(idx[np.argmin(a[idx]/np.maximum(d[idx]-1,1)/np.maximum(d[idx],1)/n[idx])]);d[j]-=1
    remain=B-int(np.dot(n,d))
    while True:
        idx=np.where((d<K)&(n<=remain))[0]
        if len(idx)==0:break
        gain=a[idx]/(d[idx]*(d[idx]+1)*n[idx])
        j=int(idx[np.argmax(gain)]);d[j]+=1;remain-=int(n[j])
    return tuple(int(v) for v in d)

def allocate_heuristic(method,n,K,B,energy):
    n=np.asarray(n);K=np.asarray(K)
    # Uniform: d_j approximately constant. Cardinality: d_j proportional
    # to sqrt(N_j). Spectral mass: d_j proportional to summed eigen-energy.
    # All three are projected to the exact common parameter budget.
    if method=='uniform':a=n.astype(float)
    elif method=='cardinality':a=n.astype(float)**2
    elif method=='spectral_mass':a=n.astype(float)*np.maximum(energy,1e-12)**2
    else:raise ValueError(method)
    return allocate_ejor(a,n,K,B)

def posterior_tail_scores(draws,names,K,xi):
    # eig draws shape (chain, retained, features, max_K); no beta!=1 draws.
    out=[];rows=[]
    for j,nam in enumerate(names):
        kj=int(K[j]);jj=[]
        for d in range(1,kj+1):
            lost=draws[:,:,j,d:kj].sum(axis=-1).reshape(-1)
            mean=float(np.mean(lost));sd=float(np.std(lost,ddof=1))
            score=mean+xi*sd
            jj.append(score)
            ci=np.quantile(lost,[.025,.975]);rows.append({'predictor':nam,'dimension':d,
              'posterior_tail_mean':mean,'posterior_tail_sd':sd,'posterior_tail_score':score,
              'posterior_tail_ci_low':float(ci[0]),'posterior_tail_ci_high':float(ci[1])})
        out.append(jj)
    return out,pd.DataFrame(rows)

def knapsack_exact(scores,levels,K,B):
    # Sparse dynamic programming over reachable *integer* embedding costs.
    # One option per predictor; objective additive. No discretization of B.
    dp={0:(0.0,())}
    for j in range(len(levels)):
        new={}
        for cost,(val,path) in dp.items():
            for d in range(1,int(K[j])+1):
                nc=cost+int(levels[j])*d
                if nc>B:continue
                nv=val+scores[j][d-1]
                if nc not in new or nv<new[nc][0]-1e-12 or (abs(nv-new[nc][0])<=1e-12 and (path+(d,))<new[nc][1]):
                    new[nc]=(nv,path+(d,))
        if not new:raise RuntimeError('Budget infeasible')
        # Pareto pruning: remove higher-cost states with no strict loss improvement.
        best=np.inf;pruned={}
        for c,(val,path) in sorted(new.items()):
            if val<best-1e-12:pruned[c]=(val,path);best=val
        dp=pruned
    best=min(dp.items(),key=lambda z:(z[1][0],z[0]))
    return tuple(best[1][1])

# ---------- Posterior: Bernoulli likelihood, exact PT + Haar orientation ----------
# This is the classification version of the manuscript's fixed-map hierarchy.
# It differs from the paper's Gaussian-response likelihood and MUST be documented.
AS,BS=2.,1.; A0,B0=2.,2.; LAMBDA=4.; AK,BK=2.,1.;SDG,SDMU=2.5,2.5

def qr_haar(A):
    q,r=np.linalg.qr(A,mode='reduced');sgn=np.where(np.diag(r)>=0,1.,-1.)
    return q*sgn

def log_beta_on_logit(z,a,b):
    if a<=0 or b<=0:return -np.inf
    return float(a*log_expit(z)+b*log_expit(-z)-betaln(a,b))
def log_gamma_on_log(z,a,b):
    if not np.isfinite(z) or abs(z)>50:return -np.inf
    return float(a*z-b*np.exp(z)+a*np.log(b)-gammaln(a))
def hypvals(h):return expit(h['z0']),expit(h['zj']),np.exp(h['lk'])
def loghyp(h,states):
    h0,hj,kappa=hypvals(h)
    if not np.isfinite(kappa) or kappa>1e5:return -np.inf
    out=log_beta_on_logit(h['z0'],A0,B0)+log_gamma_on_log(h['lk'],AK,BK)
    for j,st in enumerate(states):
        out+=log_beta_on_logit(h['zj'][j],LAMBDA*h0,LAMBDA*(1-h0))
        aa=kappa*hj[j];bb=kappa*(1-hj[j])
        for v in st['eta']:out+=log_beta_on_logit(v,aa,bb)
    return float(out)

def spectrum(st):
    sq=[np.exp(st['x'])]
    for eta in st['eta']:sq.append(sq[-1]*expit(eta))
    return np.asarray(sq)

# Implicit Householder bases for subspaces orthogonal to pi_j (without N_j^2 matrices).
HOUSEHOLDER=[]
def configure_weighted_stiefel(pis):
    global HOUSEHOLDER
    HOUSEHOLDER=[]
    for pi in pis:
        v=np.asarray(pi,dtype=float)/np.linalg.norm(pi)
        w=-v.copy();w[0]+=1.
        norm=np.linalg.norm(w)
        if norm<1e-10:raise ValueError('Degenerate categorical weights')
        HOUSEHOLDER.append(w/norm)

def householder_map(reduced,j):
    # H_j Q = [0;Q] - 2w (w[1:]^T Q), exactly pi_j^T H_j Q = 0.
    w=HOUSEHOLDER[j]
    full=np.vstack([np.zeros((1,reduced.shape[1])),reduced])
    return full-2*np.outer(w,w[1:]@reduced)

def householder_inverse(full,j):
    # H_j^T u = last N-1 coordinates of the full Householder reflection.
    w=HOUSEHOLDER[j]
    return (full-2*np.outer(w,w@full))[1:]

def matrix_from_state(st,j):
    u=householder_map(qr_haar(st['A']),j)
    v=qr_haar(st['B'])
    eig=spectrum(st)
    W=(u*np.sqrt(eig)[None,:])@v.T
    return np.vstack([np.zeros((1,W.shape[1])),W]),eig

def ref_parameters(model,levels,pis):
    emb=[]
    for j,e in enumerate(model.emb):
        W=e.weight.detach().cpu().numpy()[1:].astype(float)
        pi=np.asarray(pis[j],dtype=float)
        emb.append(W-(pi@W)[None,:])
    # MLP 8 -> 32 -> 8 -> 1, ReLU and dropout OFF at inference.
    linears=[layer for layer in model.phi if isinstance(layer,nn.Linear)]
    assert len(linears)==len(CFG.hidden_dims)+1==3
    matrices=[layer.weight.detach().cpu().numpy().T.astype(float) for layer in linears]
    offsets=[layer.bias.detach().cpu().numpy().astype(float) for layer in linears]
    g=model.gamma.weight.detach().cpu().numpy().T.astype(float)
    mu=model.gamma.bias.detach().cpu().numpy().astype(float)
    return dict(emb=emb,Ws=matrices,bs=offsets,g=g,mu=mu,
                pis=[np.asarray(x,dtype=float).copy() for x in pis])

class ExactPosterior:
    def __init__(self,ref,data,K,checkpoint):
        self.ref=ref;self.data=data;self.C=data[0];self.U=data[1].astype(float);self.y=data[2].astype(float)
        self.levels=[len(v) for v in ref['emb']];self.K=np.asarray(K)
        configure_weighted_stiefel(ref['pis'])
        self.rng=np.random.default_rng(CFG.seed+117)
        self.checkpoint=checkpoint
        # Preserve the temperature ladder of existing Olist checkpoints; SIM2's
        # geometric ladder would make an old replica checkpoint incompatible.
        self.beta=np.linspace(1.,CFG.pt_min_beta,CFG.pt_temps)
        self._save_clock=time.monotonic()
        self._progress_clock=time.monotonic()
        self._start_clock=self._progress_clock
        self._ll_calls=0
        self._iter_label='initialization'
        self._chain_label='initialization'
        self._temperature_label='initialization'
        self.chain_results=[]
        self.Ws=ref['Ws'];self.bs=ref['bs']
        assert len(self.Ws)==3
        self.ll_device=torch.device('cuda:0') if (USE_CUDA and CFG.gpu_posterior_likelihood) else torch.device('cpu')
        self.use_gpu_ll=self.ll_device.type=='cuda'
        # Bayesian transitions, stochastic proposals, SVDs and QR remain on CPU.
        # Only the frozen Bernoulli map and likelihood sum use the GPU, in FP64.
        if self.use_gpu_ll:
            self.gpu_Ws=[torch.as_tensor(w,dtype=torch.float64,device=self.ll_device).contiguous() for w in self.Ws]
            self.gpu_bs=[torch.as_tensor(b,dtype=torch.float64,device=self.ll_device) for b in self.bs]
            self.gpu_y=torch.as_tensor(self.y,dtype=torch.float64,device=self.ll_device)
            # Keep static inputs and category indices on GPU. Proposals transfer only
            # the small embedding matrix or regression coefficients, not all N x p_z
            # latent rows, for each likelihood evaluation.
            self.gpu_U=torch.as_tensor(self.U,dtype=torch.float64,device=self.ll_device)
            self.gpu_C=[torch.as_tensor(self.C[:,j],dtype=torch.long,device=self.ll_device)
                        for j in range(len(self.levels))]
            # Verify likelihood numerical equivalence on a deterministic probe
            # before allowing the GPU evaluator to affect MCMC accept/reject decisions.
            verification_rng=np.random.default_rng(20261109)
            max_difference=0.
            for zprobe in (np.zeros((len(self.y),self.Ws[0].shape[0]),dtype=np.float64),
                           verification_rng.normal(scale=.4,size=(len(self.y),self.Ws[0].shape[0]))):
                reference=self._ll_numpy(zprobe)
                accelerated=self._ll_cuda(zprobe)
                tol=max(1e-7,1e-10*abs(reference))
                max_difference=max(max_difference,abs(accelerated-reference))
                if not np.isfinite(accelerated) or abs(accelerated-reference)>tol:
                    raise AssertionError(f'CUDA posterior likelihood differs from NumPy by {abs(accelerated-reference):.3g}, tolerance {tol:.3g}')
            print(f'Bayesian frozen-map likelihood: GPU FP64 verified (max |difference|={max_difference:.3g}).',flush=True)
        else:
            print('Bayesian frozen-map likelihood: NumPy CPU (automatic fallback).',flush=True)
    def _ll_numpy(self,Z):
        h=np.maximum(0.,Z@self.Ws[0]+self.bs[0])
        h=np.maximum(0.,h@self.Ws[1]+self.bs[1])
        logits=(h@self.Ws[2]+self.bs[2]).reshape(-1)
        return float(np.sum(self.y*logits-np.logaddexp(0.,logits)))
    @torch.no_grad()
    def _ll_cuda_tensor(self,z):
        # Identical FP64 Bernoulli target; z already resides on CUDA.
        h=torch.relu(z@self.gpu_Ws[0]+self.gpu_bs[0])
        h=torch.relu(h@self.gpu_Ws[1]+self.gpu_bs[1])
        logits=(h@self.gpu_Ws[2]+self.gpu_bs[2]).reshape(-1)
        return float(torch.sum(self.gpu_y*logits-F.softplus(logits)).item())
    @torch.no_grad()
    def _ll_cuda(self,Z):
        return self._ll_cuda_tensor(torch.as_tensor(
            np.asarray(Z,dtype=np.float64),device=self.ll_device))
    def _note_ll_eval(self):
        self._ll_calls+=1
        now=time.monotonic()
        if now-self._progress_clock>=PROGRESS_INTERVAL_SECONDS:
            print(f'  [MCMC] iteration={self._iter_label} | '
                  f'chain={self._chain_label} | temperature={self._temperature_label} | '
                  f'likelihood calls={self._ll_calls:,}',flush=True)
            self._progress_clock=now
    @torch.no_grad()
    def _gpu_ll_embedding(self,gpu_base,W,j):
        self._note_ll_eval()
        # Only transfer N_j x p_z weights; category codes are GPU-resident.
        weights=torch.as_tensor(np.ascontiguousarray(W),dtype=torch.float64,
                                device=self.ll_device)
        return self._ll_cuda_tensor(gpu_base + weights.index_select(0,self.gpu_C[j]))
    @torch.no_grad()
    def _gpu_ll_global(self,gpu_base,g,mu):
        self._note_ll_eval()
        gg=torch.as_tensor(np.ascontiguousarray(g),dtype=torch.float64,device=self.ll_device)
        mm=torch.as_tensor(np.ascontiguousarray(mu),dtype=torch.float64,device=self.ll_device)
        return self._ll_cuda_tensor(gpu_base+self.gpu_U@gg+mm)
    def ll(self,Z):
        self._note_ll_eval()
        return self._ll_cuda(Z) if self.use_gpu_ll else self._ll_numpy(Z)
    def new_state(self, rng,spread=1.0):
        spectral=[];theta=[]
        for j,W in enumerate(self.ref['emb']):
            K=int(self.K[j]);u,sv,vt=np.linalg.svd(W,full_matrices=False)
            sq=np.maximum(sv[:K]**2,1e-9)
            # Match the typical Gaussian auxiliary radial scale without changing
            # QR orientations: improves burn-in over tiny-norm SVD initializers.
            A=np.sqrt(len(W)-1)*householder_inverse(u[:,:K],j)+(.10*spread)*rng.normal(size=(len(W)-1,K))
            B=np.sqrt(self.ref['g'].shape[1])*vt[:K].T+(.10*spread)*rng.normal(size=(self.ref['g'].shape[1],K))
            eta=np.log(np.clip(sq[1:]/sq[:-1],1e-6,1-1e-6))-np.log1p(-np.clip(sq[1:]/sq[:-1],1e-6,1-1e-6))
            st=dict(A=A,B=B,x=float(np.log(sq[0])+rng.normal(scale=.1*spread)),
                    eta=eta+rng.normal(scale=.2*spread,size=K-1))
            spectral.append(st);theta.append(matrix_from_state(st,j)[0])
        g=self.ref['g'].copy()+rng.normal(size=self.ref['g'].shape)*.01*spread
        mu=self.ref['mu'].copy()+rng.normal(size=self.ref['mu'].shape)*.01*spread
        Z=self.U@g+mu
        for j in range(len(spectral)):Z+=theta[j][self.C[:,j]]
        h=dict(z0=float(np.log(.5/.5)),zj=rng.normal(scale=.15,size=len(spectral)),lk=float(np.log(2.)))
        return dict(s=spectral,theta=theta,g=g,mu=mu,Z=Z,ll=self.ll(Z),h=h)
    def ess(self,x,calc,rng):
        nu=rng.normal(size=x.shape);threshold=calc(x)+math.log(rng.random())
        angle=rng.uniform(0.,2*np.pi);left=angle-2*np.pi;right=angle
        for i in range(300):
            candidate=x*np.cos(angle)+nu*np.sin(angle)
            if calc(candidate)>=threshold:return candidate
            if angle<0:left=angle
            else:right=angle
            angle=rng.uniform(left,right)
        raise RuntimeError('ESS max bracket evaluations exceeded')
    def slice1(self,x,calc,rng,w=.8):
        fx=calc(x)
        if not np.isfinite(fx):raise FloatingPointError('Slice current state invalid')
        level=fx+np.log(rng.random());left=x-w*rng.random();right=left+w
        # Randomized finite stepping-out as in Neal's invariant slice kernel.
        # Fixed symmetric caps can compromise reversibility for bounded brackets.
        m=20;J=int(rng.integers(m));K=m-1-J
        while J>0 and calc(left)>level:
            left-=w;J-=1
        while K>0 and calc(right)>level:
            right+=w;K-=1
        for i in range(250):
            xx=rng.uniform(left,right)
            if calc(xx)>=level:return float(xx)
            if xx<x:left=xx
            else:right=xx
        raise RuntimeError('Slice failed')
    def step(self,p,beta,rng,hot=False):
        # The MCMC kernels and targets are unchanged. Only rejected likelihood
        # proposals are evaluated without constructing/uploading full N x p_z Z.
        m=len(p['s'])
        order=rng.permutation(m)
        if hot:order=order[:max(1,int(np.ceil(m/2)))]
        for j in order:
            j=int(j);st=p['s'][j];oldw=p['theta'][j];codes=self.C[:,j]
            base=p['Z']-oldw[codes]
            gpu_base=(torch.as_tensor(base,dtype=torch.float64,device=self.ll_device)
                      if self.use_gpu_ll else None)
            na=st['A'].size;shapeA=st['A'].shape;shapeB=st['B'].shape
            current=np.r_[st['A'].ravel(),st['B'].ravel()]
            accepted={}
            def evaluate(v):
                ss=dict(A=v[:na].reshape(shapeA),B=v[na:].reshape(shapeB),x=st['x'],eta=st['eta'])
                w,_=matrix_from_state(ss,j)
                ll=(self._gpu_ll_embedding(gpu_base,w,j) if self.use_gpu_ll
                    else self.ll(base+w[codes]))
                accepted.update(w=w,ll=ll)
                return beta*ll
            next_=self.ess(current,evaluate,rng)
            st['A']=next_[:na].reshape(shapeA);st['B']=next_[na:].reshape(shapeB)
            p['theta'][j]=accepted['w']
            p['Z']=base+accepted['w'][codes]
            p['ll']=accepted['ll']
            # Spectral slice steps continue to target exactly the same posterior.
            for k in range(len(st['eta'])+1):
                prev=st['x'] if k==0 else st['eta'][k-1]
                _,hj,kappa=hypvals(p['h']);aa=kappa*hj[j];bb=kappa*(1-hj[j])
                base=p['Z']-p['theta'][j][codes]
                if self.use_gpu_ll:
                    gpu_base=torch.as_tensor(base,dtype=torch.float64,device=self.ll_device)
                last={}
                def target(v):
                    ss=dict(A=st['A'],B=st['B'],x=(v if k==0 else st['x']),
                             eta=st['eta'].copy())
                    if k>0:ss['eta'][k-1]=v
                    lw,_=matrix_from_state(ss,j)
                    lpr=(log_gamma_on_log(v,AS,BS) if k==0 else log_beta_on_logit(v,aa,bb))
                    ll=(self._gpu_ll_embedding(gpu_base,lw,j) if self.use_gpu_ll
                        else self.ll(base+lw[codes]))
                    last.update(w=lw,ll=ll)
                    return lpr+beta*ll
                new=self.slice1(prev,target,rng,w=.4 if k==0 else 1.3)
                if k==0:st['x']=new
                else:st['eta'][k-1]=new
                p['theta'][j]=last['w']
                p['Z']=base+last['w'][codes]
                p['ll']=last['ll']
        # Gaussian joint ESS for (gamma, mu), with the same exact normal prior.
        x=np.r_[p['g'].ravel()/SDG,p['mu'].ravel()/SDMU];ng=p['g'].size
        accepted={};base=p['Z']-self.U@p['g']-p['mu']
        gpu_base=(torch.as_tensor(base,dtype=torch.float64,device=self.ll_device)
                  if self.use_gpu_ll else None)
        def glob(v):
            g=v[:ng].reshape(p['g'].shape)*SDG;mu=v[ng:]*SDMU
            ll=(self._gpu_ll_global(gpu_base,g,mu) if self.use_gpu_ll
                else self.ll(base+self.U@g+mu))
            accepted.update(g=g,mu=mu,ll=ll)
            return beta*ll
        self.ess(x,glob,rng)
        p['g']=accepted['g'];p['mu']=accepted['mu']
        p['Z']=base+self.U@p['g']+p['mu']
        p['ll']=accepted['ll']
        # Hyperparameter slices are unchanged and do not evaluate the likelihood.
        h=p['h']
        h['z0']=self.slice1(h['z0'],lambda v:loghyp(dict(z0=v,zj=h['zj'],lk=h['lk']),p['s']),rng)
        for j in range(m):
            def hz(v):
                arr=h['zj'].copy();arr[j]=v
                return loghyp(dict(z0=h['z0'],zj=arr,lk=h['lk']),p['s'])
            h['zj'][j]=self.slice1(h['zj'][j],hz,rng)
        h['lk']=self.slice1(h['lk'],lambda v:loghyp(dict(z0=h['z0'],zj=h['zj'],lk=v),
                                                    p['s']),rng)
        return p
    def diagnostics(self,draws):
        if not draws or len(draws[0])<20:return pd.DataFrame(),False
        # chains x draws x features x eig
        spectral=np.array([[row['eig'] for row in chain] for chain in draws],float)
        hbar=np.array([[row['bar'] for row in chain] for chain in draws]);
        hglobal=np.array([[row['bar0'] for row in chain] for chain in draws]);
        hk=np.array([[row['kappa'] for row in chain] for chain in draws]);
        ll=np.array([[row['ll'] for row in chain] for chain in draws]);
        gam=np.array([[row['g'] for row in chain] for chain in draws]);
        mu=np.array([[row['mu'] for row in chain] for chain in draws]);
        tracked={}
        for j in range(spectral.shape[2]):
            for k in range(int(self.K[j])):
                tracked[f'{CATS[j]}:s2_{k+1}']=spectral[:,:,j,k]
            for d in range(1,int(self.K[j])):
                tracked[f'{CATS[j]}:tail_{d}']=spectral[:,:,j,d:int(self.K[j])].sum(-1)
            tracked[f'{CATS[j]}:bar_rho']=hbar[:,:,j]
        for k in range(gam.shape[2]):
            for j in range(gam.shape[3]):tracked[f'gamma_{k}_{j}']=gam[:,:,k,j]
        for j in range(mu.shape[2]):tracked[f'mu_{j}']=mu[:,:,j]
        tracked['hyper:bar0']=hglobal;tracked['hyper:kappa']=hk;tracked['log_bernoulli_like']=ll
        rows=[];passed=True
        for key,x in tracked.items():
            try:
                rr=float(az.rhat(x,method='rank'));bulk=float(az.ess(x,method='bulk'))
                tail=float(az.ess(x,method='tail',prob=(0.05,0.95)))
            except Exception:rr=bulk=tail=np.nan
            good=bool(np.isfinite(rr) and np.isfinite(bulk) and np.isfinite(tail) and
                      rr<=CFG.rhat_limit and bulk>=CFG.ess_bulk_min and tail>=CFG.ess_tail_min)
            passed=passed and good
            rows.append(dict(parameter=key,rhat=rr,bulk_ess=bulk,tail_ess=tail,pass_gate=good))
        return pd.DataFrame(rows),passed
    def run(self):
        cp=self.checkpoint
        if cp.exists():
            state=torch.load(cp,map_location='cpu',weights_only=False)
            assert state['fingerprint']==self.fingerprint(),'Checkpoint mismatch.'
            ladders=state['ladders'];it=state['it'];stored=state['stored']
            print(f'Resuming ALL {CFG.pt_chains} independent PT ladders, their RNGs, replicas, and cold draws at iteration {it}',flush=True)
            if len(ladders)!=CFG.pt_chains or len(stored)!=CFG.pt_chains:
                raise RuntimeError('Incompatible PT checkpoint: chain count changed')
            for lad in ladders:
                if len(lad['reps'])!=CFG.pt_temps:
                    raise RuntimeError('Incompatible PT checkpoint: temperature count changed')
                if 'labels' not in lad:
                    # Older Olist checkpoints did not monitor replica roundtrips.
                    # Start tracking from this checkpoint without inventing history.
                    lad['labels']=np.arange(CFG.pt_temps,dtype=int)
                    lad['seen_hot']=np.zeros(CFG.pt_temps,dtype=bool)
                    lad['roundtrips']=0
                    lad['roundtrip_tracking_from_iteration']=it
            old_diag,old_pass=self.diagnostics(stored)
            if old_pass and it>=CFG.pt_base:
                print(f'  [MCMC RESUME] existing posterior already PASSED strict all-parameter Rhat/ESS gate at iter {it}.',flush=True)
                return (np.array([[r['eig'] for r in ch] for ch in stored]),old_diag,True)
            if it>=CFG.pt_max:
                if CFG.require_bayes_convergence:
                    raise RuntimeError('Checkpoint reached pt_max without convergence. Increase CFG.pt_max '+
                                       'to continue this SAME posterior without clearing its history.')
                print('  [DIAGNOSTIC-ONLY] Previous run stopped at pt_max without passing diagnostics; '
                      'retaining FAIL status and reusing draws.',flush=True)
                return (np.array([[r['eig'] for r in ch] for ch in stored]),old_diag,False)

        else:
            ladders=[]
            for ci in range(CFG.pt_chains):
                rng=np.random.default_rng(CFG.seed+2400+ci)
                reps=[self.new_state(rng,spread=1.+.2*ci) for _ in self.beta]
                ladders.append(dict(rng=rng,reps=reps,swaps=np.zeros(CFG.pt_temps-1,int),
                                    trials=np.zeros(CFG.pt_temps-1,int),
                                    labels=np.arange(CFG.pt_temps,dtype=int),
                                    seen_hot=np.zeros(CFG.pt_temps,dtype=bool),
                                    roundtrips=0,roundtrip_tracking_from_iteration=0))
            stored=[[] for _ in range(CFG.pt_chains)];it=0
        # Continue at the NEXT diagnostics milestone after a saved iteration,
        # rather than re-testing on every subsequent iteration after RESUME.
        until=(CFG.pt_base if it<CFG.pt_base else
               min(CFG.pt_max, CFG.pt_base +
                   ((it-CFG.pt_base)//CFG.pt_extend+1)*CFG.pt_extend))
        while it<CFG.pt_max:
            it+=1
            self._iter_label=it
            for ci,lad in enumerate(ladders):
                self._chain_label=f'{ci+1}/{CFG.pt_chains}'
                rng=lad['rng']
                for ti,beta in enumerate(self.beta):
                    self._temperature_label=f'{ti+1}/{CFG.pt_temps}'
                    if ti==0 or it%CFG.pt_hot_update_every==0:
                        self.step(lad['reps'][ti],float(beta),rng,hot=(ti>0))
                for ti in (range(0,CFG.pt_temps-1,2) if it%2==0 else range(1,CFG.pt_temps-1,2)):
                    x,y=lad['reps'][ti],lad['reps'][ti+1]
                    ratio=float((self.beta[ti]-self.beta[ti+1])*(y['ll']-x['ll']))
                    lad['trials'][ti]+=1
                    if math.log(rng.random())<min(0.,ratio):
                        lad['reps'][ti],lad['reps'][ti+1]=y,x
                        lad['labels'][ti],lad['labels'][ti+1]=lad['labels'][ti+1],lad['labels'][ti]
                        lad['swaps'][ti]+=1
                lad['seen_hot'][int(lad['labels'][-1])]=True
                cold_label=int(lad['labels'][0])
                if lad['seen_hot'][cold_label]:
                    lad['roundtrips']+=1
                    lad['seen_hot'][cold_label]=False
                if it>CFG.pt_burn and (it-CFG.pt_burn)%CFG.pt_thin==0:
                    cold=lad['reps'][0]
                    eig=np.zeros((len(CATS),max(self.K)))
                    for j,ss in enumerate(cold['s']):eig[j,:int(self.K[j])]=spectrum(ss)
                    h0,hj,kap=hypvals(cold['h'])
                    stored[ci].append(dict(eig=eig,bar=hj.copy(),bar0=float(h0),kappa=float(kap),
                                           ll=cold['ll'],g=cold['g'].copy(),mu=cold['mu'].copy()))
            if it in (1,10) or it%1000==0:
                # SIM2-style full cached-likelihood/latent-state consistency audit.
                for lad in ladders:
                    for replica in lad['reps']:
                        reconstructed=self.U@replica['g']+replica['mu']
                        for j in range(len(self.levels)):
                            reconstructed+=replica['theta'][j][self.C[:,j]]
                        if not np.allclose(reconstructed,replica['Z'],rtol=1e-10,atol=1e-8):
                            raise AssertionError('MCMC latent predictor cache is inconsistent')
                        check_ll=self.ll(reconstructed)
                        if not np.isclose(check_ll,replica['ll'],rtol=1e-11,atol=1e-7):
                            raise AssertionError('MCMC cached Bernoulli likelihood is inconsistent')
                # Cache audit passed; log only a failure (the assertions above).
            if it%MCMC_ITER_REPORT_EVERY==0:print(f'MCMC iter={it}, retained/chain={len(stored[0])}, '
                              f'cold LL={[round(l["reps"][0]["ll"],1) for l in ladders]} | '
                              f'roundtrips={[int(l["roundtrips"]) for l in ladders]}',flush=True)
            if (time.monotonic()-self._save_clock)>=CFG.checkpoint_minutes*60:
                self._atomic_save(dict(fingerprint=self.fingerprint(),ladders=ladders,it=it,stored=stored))
                report('MCMC CHECKPOINT',f'iteration {it} | {CFG.pt_chains} cold chains retained={len(stored[0])} | saved {cp}')
            if it>=until:
                diag,ok=self.diagnostics(stored)
                diag.to_csv(POST/'mcmc_diagnostics_intermediate.csv',index=False)
                print('Gate:',('PASS' if ok else 'FAIL'),'at',it,'| failing:',
                      int((~diag.pass_gate).sum()) if len(diag) else 'NA',flush=True)
                if len(diag):
                    worst=diag.loc[~diag.pass_gate,['parameter','rhat','bulk_ess','tail_ess']]
                    # Detailed parameter diagnostics are saved in the CSV only.
                self._atomic_save(dict(fingerprint=self.fingerprint(),ladders=ladders,it=it,stored=stored))
                if ok:break
                until=min(CFG.pt_max,it+CFG.pt_extend)
                if it>=CFG.pt_max:break
        diag,ok=self.diagnostics(stored)
        diag.to_csv(POST/'mcmc_diagnostics_final.csv',index=False)
        self._atomic_save(dict(fingerprint=self.fingerprint(),ladders=ladders,it=it,stored=stored))
        np.savez_compressed(POST/'cold_eigenvalue_draws.npz',
             eig=np.array([[r['eig'] for r in ch] for ch in stored]),
             rhat_pass=np.asarray(ok))
        with open(POST/'posterior_status.json','w') as f:
            json.dump(dict(converged=bool(ok),iterations=it,retained_per_chain=len(stored[0]),
                 min_bulk_ess=float(diag.bulk_ess.min()) if len(diag) else None,
                 max_rhat=float(diag.rhat.max()) if len(diag) else None,
                 swap_acceptances=[(l['swaps']/np.maximum(l['trials'],1)).tolist() for l in ladders],
                 roundtrips_since_tracking=[int(l['roundtrips']) for l in ladders],
                 roundtrip_tracking_from_iteration=[int(l['roundtrip_tracking_from_iteration']) for l in ladders],
                 posterior_training_n=len(self.y),
                 warning='Exact for stated Bernoulli likelihood, reference map, prior and chosen training subset; not an independent full-data posterior'),f,indent=2)
        return np.array([[r['eig'] for r in ch] for ch in stored]),diag,ok
    def fingerprint(self):
        settings=asdict(CFG)
        for editable in ('pt_base','pt_extend','pt_max','checkpoint_minutes','require_bayes_convergence','bootstrap_reps','plot_dpi'):
            settings.pop(editable,None)
        d=dict(implementation='gpu-auto-fp64-v3',CFG=settings,train_size=len(self.y),levels=self.levels,
               reference_digest=hashlib.sha256(np.concatenate(
                [np.asarray(x).ravel() for x in (self.ref['emb']+self.ref['Ws']+self.ref['bs']+
                [self.ref['g'],self.ref['mu']]+self.ref['pis'])]
             ).tobytes()).hexdigest())
        return hashlib.sha256(json.dumps(d,sort_keys=True,default=str).encode()).hexdigest()
    def _atomic_save(self,state):
        tmp=self.checkpoint.with_suffix('.tmp');torch.save(state,tmp);os.replace(tmp,self.checkpoint)
        self._save_clock=time.monotonic()

# ---------- Evaluations and manuscript-ready outputs ----------
def cal_bins(y,p,n_bins=10):
    ids=np.clip(np.searchsorted(np.linspace(0,1,n_bins+1)[1:-1],p,side='right'),0,n_bins-1)
    result=[]
    for b in range(n_bins):
        mask=ids==b
        if mask.any():result.append(dict(bin=b,n=int(mask.sum()),prob=float(p[mask].mean()),rate=float(y[mask].mean())))
    return pd.DataFrame(result)
def metric_dict(y,p,extended=False):
    pred=(p>=.5).astype(int)
    cb=cal_bins(y,p);abs_gap=np.abs(cb.rate-cb.prob)
    ece=float((cb.n/len(y)*abs_gap).sum());mce=float(abs_gap.max())
    out=dict(auc=roc_auc_score(y,p),average_precision=average_precision_score(y,p),
        logloss=log_loss(y,p),brier=brier_score_loss(y,p),accuracy=accuracy_score(y,pred),
        precision=precision_score(y,pred,zero_division=0),recall=recall_score(y,pred,zero_division=0),
        f1=f1_score(y,pred,zero_division=0),balanced_accuracy=balanced_accuracy_score(y,pred),
        mcc=matthews_corrcoef(y,pred),ece=ece,mce=mce)
    if extended:
        tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
        out.update(tn=int(tn),fp=int(fp),fn=int(fn),tp=int(tp),
                   specificity=float(tn/(tn+fp)) if tn+fp else np.nan,
                   negative_predictive_value=float(tn/(tn+fn)) if tn+fn else np.nan)
        try:
            logits=np.log(np.clip(p,1e-7,1-1e-7)/np.clip(1-p,1e-7,1-1e-7)).reshape(-1,1)
            cal=LogisticRegression(penalty=None,solver='lbfgs',max_iter=300).fit(logits,y)
            out['calibration_intercept']=float(cal.intercept_[0]);out['calibration_slope']=float(cal.coef_[0][0])
        except Exception:
            out['calibration_intercept']=np.nan;out['calibration_slope']=np.nan
    return out
def savefig(name):
    plt.tight_layout();plt.savefig(FIG/(name+'.png'),dpi=CFG.plot_dpi,bbox_inches='tight')
    plt.savefig(FIG/(name+'.pdf'),bbox_inches='tight');plt.close()
def save_table(df,name):
    df.to_csv(TAB/(name+'.csv'),index=False)
    try:
        with open(TAB/(name+'.tex'),'w') as f:
            f.write(df.to_latex(index=False,float_format=lambda v:f'{v:.4f}',escape=True))
    except Exception as exc:print('LaTex table write failed:',exc)

def method_allocations(levels,K,eigs,a,energy,B,bayesian_uncertainty,bayesian_mean):
    # Every competing embedding method has the SAME feasible set D_B and cost.
    # EJOR baseline: closed-form allocation with integer projection.
    d={}
    if bayesian_uncertainty is not None:
        d['bayesian_uncertainty']=knapsack_exact(bayesian_uncertainty,levels,K,B)
    if bayesian_mean is not None:
        d['bayesian_mean']=knapsack_exact(bayesian_mean,levels,K,B)
    d['ejor_closed_form']=allocate_ejor(a,levels,K,B)
    for meth in ('uniform','cardinality','spectral_mass'):
        n=np.asarray(levels,dtype=float)
        if meth=='uniform':aa=n
        elif meth=='cardinality':aa=n**2
        else:aa=n*np.maximum(energy,1e-12)**2
        d[meth]=knapsack_exact([[float(aa[j])/q for q in range(1,int(K[j])+1)]
                               for j in range(len(levels))],levels,K,B)
    for method,di in d.items():
        assert len(di)==len(levels)
        assert all(1<=v<=int(k) and v<int(n) for v,k,n in zip(di,K,levels))
        assert embedding_cost(levels,di)<=B
    return d

def _make_model(method,dims,levels):
    if method=='onehot':return OneHotMLP(levels,len(NUMS))
    if method=='numeric_only':return NumericMLP(len(NUMS))
    return EmbeddingMLP(levels,dims,len(NUMS))

def _dim_key(dims):
    return 'd_'+'_'.join(str(int(v)) for v in dims)

def evaluate_inner_dimensions(dim,levels,inner,iv,run_fingerprint,progress_label=None):
    # Cache keyed by the actual architecture, since different allocation
    # methods may produce identical vectors at one or more budgets.
    key=_dim_key(dim);fp=hashlib.sha256((run_fingerprint+'|'+key).encode()).hexdigest()[:18]
    path=FITCACHE/(fp+'.json');history_path=FITCACHE/(fp+'.csv')
    if path.exists() and history_path.exists():
        info=json.loads(path.read_text());hist=pd.read_csv(history_path)
        if info.get('fingerprint')==run_fingerprint and info.get('dims')==list(dim):
            return info['metrics'],hist
    seed_everything(CFG.seed+101)
    net=EmbeddingMLP(levels,dim,len(NUMS))
    net,hist,_=train_model(net,inner,iv,epochs=CFG.epochs,seed=CFG.seed+101,progress_label=progress_label)
    p=probabilities(net,iv);metrics=metric_dict(iv[2],p)
    hist.to_csv(history_path,index=False)
    payload={'fingerprint':run_fingerprint,'dims':list(dim),'metrics':metrics}
    tmp=path.with_suffix('.tmp');tmp.write_text(json.dumps(payload,indent=2))
    os.replace(tmp,path)
    del net
    return metrics,hist

def fit_final_model(method,dims,levels,final,outer,tag,run_fingerprint):
    # Fixed epoch count for ALL methods. Same seed and full development set.
    # Final refit after budget selection, heldout used solely for reporting.
    key=tag.replace('/','_')
    pred_path=PRED/(key+'_predictions.csv')
    weights_path=OUT/(key+'_weights.pt')
    history_path=TAB/(key+'_final_training_history.csv')
    fp=hashlib.sha256((run_fingerprint+'|'+str(dims)+'|'+method+'|'+str(CFG.epochs)).encode()).hexdigest()
    meta_path=OUT/(key+'_final_signature.json')
    if (pred_path.exists() and weights_path.exists() and history_path.exists() and meta_path.exists()
          and json.loads(meta_path.read_text()).get('fingerprint')==fp):
        prev=pd.read_csv(pred_path)
        p=prev['probability'].to_numpy();hist=pd.read_csv(history_path)
        net=_make_model(method,dims,levels)
        net.load_state_dict(torch.load(weights_path,map_location='cpu',weights_only=True))
    else:
        seed_everything(CFG.seed+777)
        net=_make_model(method,dims,levels)
        net,hist,_=train_model(net,final,None,epochs=CFG.epochs,seed=CFG.seed+777,progress_label=f'FINAL {tag}')
        p=probabilities(net,outer)
        hist.to_csv(history_path,index=False)
        pd.DataFrame({'order_index':np.arange(len(p)),'observed':outer[2],
                      'probability':p}).to_csv(pred_path,index=False)
        torch.save(net.cpu().state_dict(),weights_path)
        meta_path.write_text(json.dumps({'fingerprint':fp,'method':method,'dims':dims}))
    if len(p)!=len(outer[2]):raise RuntimeError('Prediction cache length mismatch')
    metrics=metric_dict(outer[2],p,extended=True)
    cost=(0 if dims is None else embedding_cost(levels,dims))
    # One-hot is an unbudgeted benchmark: categorical weight params = sum N_j * hidden_1.
    if method=='onehot':cost=int(sum(levels)*CFG.hidden_dims[0])
    row=dict(model=tag,method=method,feature_set=('numerical_only' if method=='numeric_only'
          else 'all_categorical_plus_numerical'),embedding_params=cost,
          trainable_params=count_parameters(net),trained_epochs=CFG.epochs,**metrics)
    if dims is not None:row.update({f'dim_{j+1}':int(k) for j,k in enumerate(dims)})
    return row,p,hist

def paired_bootstrap(y,base,comparators):
    rng=np.random.default_rng(CFG.seed+222)
    n=len(y);rows=[]
    for meth,p in comparators.items():
        if meth=='proposed_bayesian':continue
        delta=[]
        for _ in range(CFG.bootstrap_reps):
            idx=rng.integers(0,n,size=n)
            if len(np.unique(y[idx]))<2:continue
            delta.append(roc_auc_score(y[idx],base[idx])-roc_auc_score(y[idx],p[idx]))
        if len(delta)<10:continue
        rows.append(dict(comparator=meth,auc_difference=float(roc_auc_score(y,base)-roc_auc_score(y,p)),
              ci_low=float(np.quantile(delta,.025)),ci_high=float(np.quantile(delta,.975)),
              paired_bootstrap_p_two_sided=float(min(1,2*min(np.mean(np.array(delta)<=0),np.mean(np.array(delta)>=0)))),
              bootstrap_reps=len(delta)))
    table=pd.DataFrame(rows)
    if len(table):
        p=table.paired_bootstrap_p_two_sided.to_numpy();order=np.argsort(p);m=len(p)
        raw=(m-np.arange(m))*p[order];corrected=np.maximum.accumulate(raw).clip(0,1)
        out=np.empty(m);out[order]=corrected;table['holm_adjusted_p']=out
    return table

def make_figures(results,preds,histories,alloc_table,select,budgets,levels,eigs,taildf,outer_y):
    # 1. Main AUC vs total parameters
    plt.figure(figsize=(9,5));plt.scatter(results.trainable_params,results.auc,s=65)
    for _,r in results.iterrows():plt.annotate(str(r.model),(r.trainable_params,r.auc),fontsize=7,xytext=(3,3),textcoords='offset points')
    plt.xlabel('Trainable parameters');plt.ylabel('Held-out ROC AUC');savefig('auc_vs_parameter_count')
    # 2. PR curves and 3. ROC curves
    plt.figure(figsize=(8,6))
    main_plot_keys=['bayesian_uncertainty','bayesian_mean','ejor_closed_form','uniform','cardinality','spectral_mass','onehot','numeric_only','full_reference_budget']
    for k in main_plot_keys:
        if k not in preds:continue
        p=preds[k];pr,rc,_=precision_recall_curve(outer_y,p);plt.plot(rc,pr,label=k)
    plt.xlabel('Recall');plt.ylabel('Precision');plt.legend(fontsize=7);savefig('precision_recall_curves')
    plt.figure(figsize=(8,6))
    for k in main_plot_keys:
        if k not in preds:continue
        p=preds[k];fpr,tpr,_=roc_curve(outer_y,p);plt.plot(fpr,tpr,label=k)
    plt.plot([0,1],[0,1],'--',alpha=.5);plt.xlabel('False positive rate');plt.ylabel('True positive rate')
    plt.legend(fontsize=7);savefig('roc_curves')
    # 4. Reliability diagram
    plt.figure(figsize=(7,6))
    for k in main_plot_keys:
        if k not in preds:continue
        p=preds[k];c=cal_bins(outer_y,p);plt.plot(c.prob,c.rate,'o-',label=k,alpha=.7)
    plt.plot([0,1],[0,1],'--',alpha=.5);plt.xlabel('Mean predicted probability')
    plt.ylabel('Observed positive fraction');plt.legend(fontsize=7);savefig('reliability_diagram')
    # 5. Allocation dimensions at common budgets, 6. spectral pilot
    piv=alloc_table.pivot(index='method',columns='predictor',values='dimension')
    plt.figure(figsize=(10,4));plt.imshow(piv.values,aspect='auto');plt.xticks(range(len(piv.columns)),piv.columns,rotation=40,ha='right',fontsize=8)
    plt.yticks(range(len(piv.index)),piv.index);plt.colorbar(label='Embedding dimension')
    savefig('embedding_allocation_heatmap')
    plt.figure(figsize=(8,5))
    for j,vals in enumerate(eigs):plt.plot(range(1,len(vals)+1),vals,'o-',label=CATS[j])
    plt.xlabel('Singular component');plt.ylabel('Squared singular value (pilot)');plt.yscale('symlog',linthresh=1e-4)
    plt.legend(fontsize=7);savefig('pilot_spectra')
    # 7. posterior tail means + 95% credible intervals
    plt.figure(figsize=(9,5))
    for nam,g in taildf.groupby('predictor'):
        plt.plot(g.dimension,g.posterior_tail_mean,marker='o',label=nam)
        plt.fill_between(g.dimension.values,g.posterior_tail_ci_low.values,g.posterior_tail_ci_high.values,alpha=.12)
    plt.xlabel('Retained dimensions');plt.ylabel('Posterior omitted spectral energy')
    plt.legend(fontsize=7);savefig('posterior_spectral_tails')
    # 8. Model learning curves
    plt.figure(figsize=(9,5))
    for key,h in histories.items():
        if len(h):plt.plot(h.epoch,h.inner_val_logloss,label=key)
    plt.xlabel('Epoch');plt.ylabel('Inner-validation log loss');plt.legend(fontsize=7)
    savefig('inner_validation_learning_curves')
    # 9. Budget performance by method
    plt.figure(figsize=(9,5))
    for meth,g in select.groupby('method'):
        plt.plot(g.budget,g.inner_val_auc,'o-',label=meth)
    plt.xlabel('Embedding parameter budget');plt.ylabel('Inner-validation AUC')
    plt.legend(fontsize=7);savefig('inner_validation_budget_curves')
    # 11. Budget choice uses logloss and the manuscript's smallest-feasible rule.
    plt.figure(figsize=(9,5))
    for meth,g in select.groupby('method'):
        plt.plot(g.budget,g.inner_val_logloss,'o-',label=meth)
    plt.xlabel('Embedding parameter budget B');plt.ylabel('Inner-validation Bernoulli log loss')
    plt.legend(fontsize=7);savefig('budget_selection_logloss_vs_budget')
    # 10. Histogram of prediction probabilities
    plt.figure(figsize=(9,5))
    for key in main_plot_keys:
        if key in preds:plt.hist(preds[key],bins=np.linspace(0,1,26),alpha=.3,label=key,histtype='step',density=True)
    plt.xlabel('Predicted probability');plt.ylabel('Density');plt.legend(fontsize=7)
    savefig('prediction_probability_distributions')

def main():
    report('STAGE 1/6 DATA',f'{RUN_MODE.upper()} mode | request/reuse user Olist files; fixed split seed={CFG.seed}')
    paths=data_sources();df=build_olist(paths)
    assert 'primary_seller_id' not in df.columns and len(CATS)==6
    if CFG.smoke_test:
        _,df=train_test_split(df,test_size=min(1800/len(df),.20),stratify=df[TARGET],random_state=CFG.seed)
    assert df[TARGET].nunique()==2 and df.order_id.is_unique
    development_df,heldout_df=train_test_split(df,test_size=CFG.outer_validation_fraction,
                                     random_state=CFG.seed,stratify=df[TARGET])
    inner_df,iv_df=train_test_split(development_df,test_size=CFG.inner_validation_fraction,
                                   random_state=CFG.seed+1,stratify=development_df[TARGET])
    for f,d in [('development.csv',development_df),('heldout_test.csv',heldout_df),
                ('inner_training.csv',inner_df),('inner_budget_validation.csv',iv_df)]:
        d[['order_id',TARGET]].to_csv(OUT/f,index=False)
    prep=Prep().fit(inner_df)
    inner=prep.transform(inner_df);iv=prep.transform(iv_df)
    final=prep.transform(development_df);heldout=prep.transform(heldout_df)
    n=np.asarray(prep.levels,dtype=int);K=limits(n)
    if not np.all(n>=2):raise ValueError('Remove constant categorical predictor before allocating')
    if REFERENCE_PZ!=CFG.hidden_dims[-1]:raise AssertionError('Reference dim is not architecture derived')
    assert len(CATS)==len(n)==len(K)   # m is observed, never hardcoded
    B=budget_grid(n,K);Bmin=int(sum(n));Bmax=embedding_cost(n,K)
    assert B[0]==Bmin and B[-1]==Bmax
    run_signature=dict(seed=CFG.seed,levels=n.tolist(),K=K.tolist(),reference_pz=REFERENCE_PZ,
         cats=CATS,nums=NUMS,train_n=len(inner_df),holdout_n=len(heldout_df),
         inner_order_digest=hashlib.sha256(''.join(inner_df.order_id.astype(str)).encode()).hexdigest(),
         lr=CFG.lr,epochs=CFG.epochs,hidden_dims=CFG.hidden_dims,dropout=CFG.dropout,
         batch_size=CFG.batch_size,weight_decay=CFG.weight_decay)
    run_fingerprint=hashlib.sha256(json.dumps(run_signature,sort_keys=True).encode()).hexdigest()
    summary=dict(rows=len(df),development=len(development_df),heldout_test=len(heldout_df),
         inner_training=len(inner_df),inner_budget_validation=len(iv_df),
         outcome_positive_rate=float(df[TARGET].mean()),categories=dict(zip(CATS,n.tolist())),
         features=dict(numerical=NUMS,categorical=CATS),m=len(CATS),reference_pz=REFERENCE_PZ,
         K_j=K.tolist(),budget_min=Bmin,budget_max=Bmax,budgets=B,
         budget_rule='min B such that inner validation logloss(B) <= logloss(Bmax) + tolerance',
         epoch_training='fixed, no patience, no best-epoch weight selection',
         model=f'EmbDim.ipynb (32,8), ReLU, dropout={CFG.dropout:g}, Adam, 50 epochs',
         definition='estimated delivery date minus purchase timestamp strictly >24 days',
         split='70/30 development/test, internal 80/20 budget-validation, no CV',
         prior='fixed-dimensional hierarchical Beta spectral shrinkage',
         likelihood='Bernoulli logit with frozen reference map, NOT Gaussian response',
         posterior_rows=('full inner-training' if CFG.bayes_max_rows==0 else CFG.bayes_max_rows),
         empirical_probability_weighted_centering=True,one_dataset_split=True,
         execution_device=str(DEVICE),gpu_name=(torch.cuda.get_device_name(0) if USE_CUDA else None),
         bayesian_ll_backend=('cuda_float64' if USE_CUDA and CFG.gpu_posterior_likelihood else 'numpy_cpu'))
    (OUT/'dataset_description.json').write_text(json.dumps(summary,indent=2))
    save_table(pd.DataFrame([dict(predictor=c,N_j=int(nn),K_j=int(k))
            for c,nn,k in zip(CATS,n,K)]),'categorical_structure')
    print('DATA',len(df),'m',len(CATS),'N_j',n,'K_j',K,'p_z',REFERENCE_PZ,
          'budgets',B,flush=True)
    print('TRAINING: fixed epochs',CFG.epochs,'no early stopping',flush=True)
    # Rich reference additive model with exactly EmbDim's MLP architecture.
    ref_settings=dict(run=run_fingerprint,method='anchored_additive_reference',
                      pilot_epochs=CFG.pilot_epochs,pi_weights=hashlib.sha256(
                      np.concatenate(prep.pis).tobytes()).hexdigest())
    ref_sig=hashlib.sha256(json.dumps(ref_settings,sort_keys=True).encode()).hexdigest()
    ref_path=OUT/'reference_model_weights.pt';ref_meta=OUT/'reference_model_signature.json'
    ref=ReferenceMLP(n,len(NUMS),prep.pis)
    report('STAGE 2/6 REFERENCE','Fit or reuse the anchored reference model; Bayesian MCMC is NOT running yet')
    if ref_path.exists() or ref_meta.exists():
        if not ref_path.exists() or not ref_meta.exists():
            raise RuntimeError('Incomplete reference checkpoint')
        if json.loads(ref_meta.read_text()).get('signature')!=ref_sig:
            raise RuntimeError('Reference checkpoint configuration mismatch. Change output_dir.')
        ref.load_state_dict(torch.load(ref_path,map_location='cpu',weights_only=True))
        report('STAGE 2/6 REFERENCE','Compatible reference model loaded from checkpoint')
    else:
        ref,h,_=train_model(ref,inner,None,epochs=CFG.pilot_epochs,seed=CFG.seed+40,progress_label='REFERENCE PILOT')
        h.to_csv(TAB/'reference_training_history.csv',index=False)
        torch.save(ref.cpu().state_dict(),ref_path)
        ref_meta.write_text(json.dumps(dict(signature=ref_sig,settings=ref_settings),indent=2))
    ref.eval();eigs,a,energy=spectral_pilot(ref,n)
    save_table(pd.DataFrame({'predictor':CATS,'N_j':n,'K_j':K,
        'a_hat_ejor':a,'total_reference_spectral_energy':energy}),
        'reference_spectral_coefficients')

    if not CFG.run_exact_bayes:
        raise RuntimeError('The manuscript comparison requires an actual Bayesian MCMC posterior')
    classical_methods=('ejor_closed_form','uniform',
                       'cardinality','spectral_mass')
    bayesian_methods=('bayesian_mean','bayesian_uncertainty')
    # ---------------------------------- CLASSICAL METHODS FIRST ----------------------
    # No Bayesian MCMC or posterior draws are needed for these models. Their
    # selected-budget results remain available even if MCMC later fails.
    report('STAGE 3/6 CLASSICAL BUDGETS',
           f'Start {len(classical_methods)} non-Bayesian methods at {len(B)} budgets; '
           f'{len(classical_methods)*len(B)} candidate method/budget evaluations')
    grid_rows=[];all_allocs=[];inner_histories={};validated={}
    completed=0;total_classical=len(classical_methods)*len(B)

    def add_budget_fit(method,budget,dim):
        nonlocal completed
        completed+=1
        label=f'{method} | B={int(budget):,} | {completed}/{total_classical if method in classical_methods else total_classical+len(bayesian_methods)*len(B)}'
        metrics,h=evaluate_inner_dimensions(dim,n,inner,iv,run_fingerprint,progress_label=label)
        grid_rows.append(dict(method=method,budget=int(budget),
                 used_embedding_parameters=embedding_cost(n,dim),
                 dims=json.dumps(dim),inner_val_auc=float(metrics['auc']),
                 inner_val_logloss=float(metrics['logloss']),
                 inner_val_brier=float(metrics['brier']),inner_val_ece=float(metrics['ece']),
                 trained_epochs=CFG.epochs))
        all_allocs.extend([dict(method=method,budget=int(budget),predictor=c,
                 dimension=int(d),allocated_parameters=int(n[j]*d))
                 for j,(c,d) in enumerate(zip(CATS,dim))])
        validated[(method,int(budget))]=tuple(dim)
        inner_histories.setdefault(method,h)

    def select_budgets(tab):
        chosen_rows=[]
        for method,g in tab.groupby('method',sort=False):
            g=g.sort_values('budget')
            maximum=g[g.budget==Bmax].iloc[0]
            threshold=float(maximum.inner_val_logloss+CFG.budget_tolerance)
            candidates=g[g.inner_val_logloss<=threshold+1e-12]
            if len(candidates)==0:raise AssertionError('Bmax must be eligible')
            chosen=candidates.iloc[0]
            chosen_rows.append(dict(method=method,budget=int(chosen.budget),
                used_embedding_parameters=int(chosen.used_embedding_parameters),
                dims=chosen.dims,inner_val_auc=float(chosen.inner_val_auc),
                inner_val_logloss=float(chosen.inner_val_logloss),
                maximum_budget_logloss=float(maximum.inner_val_logloss),
                acceptance_threshold=threshold,budget_tolerance=CFG.budget_tolerance,
                embedding_saving_vs_Bmax=1-int(chosen.used_embedding_parameters)/Bmax,
                selected_rule='minimum eligible budget, NOT maximum AUC'))
        return pd.DataFrame(chosen_rows)

    for bi,budget in enumerate(B,1):
        report('CLASSICAL BUDGET',f'Budget {bi}/{len(B)}: B={budget:,}',bi,len(B))
        allocations=method_allocations(n,K,eigs,a,energy,budget,None,None)
        assert set(allocations)==set(classical_methods)
        for method in classical_methods:
            add_budget_fit(method,budget,allocations[method])
    classical_grid=pd.DataFrame(grid_rows)
    save_table(classical_grid,'classical_budget_comparison')
    save_table(pd.DataFrame(all_allocs),'classical_allocation_by_budget_long')
    classical_selected=select_budgets(classical_grid)
    save_table(classical_selected,'classical_selected_budgets')
    report('STAGE 3/6 COMPLETE','Classical budget evaluations and selections saved before Bayesian MCMC')

    # Final holdout is used exclusively for reporting, never budget selection.
    report('STAGE 4/6 CLASSICAL REFITS','Train selected classical models, full budget and unbudgeted benchmarks')
    outputs=[];preds={};histories={};allocrows=[]
    total_final=len(classical_methods)+3
    for fi,method in enumerate(classical_methods,1):
        sr=classical_selected.loc[classical_selected.method==method].iloc[0]
        budget=int(sr.budget);dims=validated[(method,budget)]
        report('CLASSICAL FINAL',f'{method}; selected B={budget:,}, dims={dims}',fi,total_final)
        row,pp,h=fit_final_model('embedding',dims,n,final,heldout,method,run_fingerprint)
        row.update(method=method,selected_budget=budget,comparison='selected_budget',
                   is_budgeted=True,convergence_certified=True,reference_dim=REFERENCE_PZ)
        outputs.append(row);preds[method]=pp;histories[method]=inner_histories[method]
        save_table(pd.DataFrame(outputs),'classical_heldout_results_partial')
    fulldim=tuple(K.tolist())
    report('CLASSICAL FINAL',f'Full embedding budget reference Bmax={Bmax:,}',len(classical_methods)+1,total_final)
    row,pp,h=fit_final_model('embedding',fulldim,n,final,heldout,'full_reference_budget',run_fingerprint)
    row.update(method='full_reference_budget',selected_budget=Bmax,comparison='reference',
               is_budgeted=True,convergence_certified=True,reference_dim=REFERENCE_PZ)
    outputs.append(row);preds['full_reference_budget']=pp;histories['full_reference_budget']=h
    for idx,meth in enumerate(('numeric_only','onehot'),len(classical_methods)+2):
        report('CLASSICAL FINAL',f'Unbudgeted benchmark: {meth}',idx,total_final)
        row,pp,h=fit_final_model(meth,None,n,final,heldout,meth,run_fingerprint)
        row.update(selected_budget=np.nan,comparison='unbudgeted_baseline',
                   is_budgeted=False,convergence_certified=True,reference_dim=REFERENCE_PZ)
        outputs.append(row);preds[meth]=pp;histories[meth]=h
    save_table(pd.DataFrame(outputs),'classical_heldout_method_comparison')
    report('STAGE 4/6 COMPLETE','All non-Bayesian baselines fitted and results saved; starting OUR method last')

    # --------------------------------- BAYESIAN METHOD LAST --------------------------
    if CFG.bayes_max_rows and len(inner_df)>CFG.bayes_max_rows:
        ii,_=train_test_split(np.arange(len(inner_df)),train_size=CFG.bayes_max_rows,
                              stratify=inner[2],random_state=CFG.seed+331)
        bayesdata=tuple(z[ii] for z in inner)
        print('WARNING: Bayesian SUBSAMPLE mode; do not report as full-data posterior.',flush=True)
    else:
        bayesdata=inner
    refpar=ref_parameters(ref,n,prep.pis)
    with torch.no_grad():
        ref=ref.to(DEVICE).eval()
        cc=torch.as_tensor(inner[0][:64],dtype=torch.long,device=DEVICE)
        xx=torch.as_tensor(inner[1][:64],dtype=torch.float32,device=DEVICE)
        torch_logits=ref(cc,xx).cpu().numpy()
    latent=inner[1][:64]@refpar['g']+refpar['mu']
    for j in range(len(CATS)):
        codes=inner[0][:64,j]
        latent+=np.where((codes>0)[:,None],refpar['emb'][j][np.maximum(codes-1,0)],0.0)
    hh=np.maximum(0.,latent@refpar['Ws'][0]+refpar['bs'][0])
    hh=np.maximum(0.,hh@refpar['Ws'][1]+refpar['bs'][1])
    numpy_logits=(hh@refpar['Ws'][2]+refpar['bs'][2]).reshape(-1)
    if not np.allclose(torch_logits,numpy_logits,atol=2e-4,rtol=3e-5):
        raise AssertionError('Frozen numpy Bayesian map disagrees with PyTorch reference')
    for j,mat in enumerate(refpar['emb']):
        assert np.max(np.abs(prep.pis[j]@mat))<1e-6
    report('STAGE 5/6 BAYESIAN MCMC',
           f'MCMC START | {CFG.pt_chains} independent chains | {CFG.pt_temps} temperatures | '
           f'{len(bayesdata[2]):,} training cases | every {CFG.checkpoint_minutes:g} min checkpoint')
    inference=ExactPosterior(refpar,bayesdata,K,POST/'posterior_checkpoint.pt')
    draws,diag,gate=inference.run()
    report('STAGE 5/6 BAYESIAN MCMC',
           f'MCMC complete | convergence gate={"PASS" if gate else "FAIL"} | ' +
           (f'max Rhat={diag.rhat.max():.4f} | min bulk ESS={diag.bulk_ess.min():.1f}'
            if len(diag) else 'diagnostics not yet estimable from retained draws'))
    score_sd,taildf=posterior_tail_scores(draws,CATS,K,CFG.posterior_xi)
    score_mean,mean_df=posterior_tail_scores(draws,CATS,K,0.0)
    save_table(taildf,'posterior_tail_uncertainty')
    save_table(mean_df,'posterior_tail_mean_only')
    if not gate and CFG.require_bayes_convergence:
        report('MCMC NOT CONVERGED','Classical results are already saved. '
               'Bayesian allocation/refits stopped until the same checkpoint is extended.')
        return

    report('STAGE 6/6 BAYESIAN ALLOCATION',
           f'Fit posterior-mean and OUR uncertainty-aware allocations at {len(B)} budgets')
    total_bayes=len(bayesian_methods)*len(B)
    for bi,budget in enumerate(B,1):
        allocations=method_allocations(n,K,eigs,a,energy,budget,score_sd,score_mean)
        for method in bayesian_methods:
            add_budget_fit(method,budget,allocations[method])
    grid=pd.DataFrame(grid_rows)
    save_table(grid,'budget_all_methods')
    save_table(pd.DataFrame(all_allocs),'allocation_by_budget_long')
    selected=select_budgets(grid)
    save_table(selected,'selected_budgets')
    commonB=int(selected.loc[selected.method=='bayesian_uncertainty','budget'].iloc[0])
    report('BAYESIAN SELECTION',f'OUR method selected B={commonB:,}; '
           f'posterior-mean B={int(selected.loc[selected.method=="bayesian_mean","budget"].iloc[0]):,}')
    # Compare every classical method at the budget selected by OUR method.
    # This supplemental matched-budget step is necessarily after Bayesian budget
    # determination, but the Bayesian final model is trained LAST.
    for mi,method in enumerate(classical_methods,1):
        dim=validated[(method,commonB)]
        for j,c in enumerate(CATS):
            allocrows.append(dict(method=method,predictor=c,dimension=int(dim[j]),
               allocated_parameters=int(n[j]*dim[j]),common_budget=commonB))
        orig_b=int(classical_selected.loc[classical_selected.method==method,'budget'].iloc[0])
        if orig_b==commonB:
            continue
        tag=method+'_common_budget'
        report('MATCHED-BUDGET BASELINE',f'{method} at OUR selected B={commonB:,}',mi,len(classical_methods))
        row,pp,h=fit_final_model('embedding',dim,n,final,heldout,tag,run_fingerprint)
        row.update(method=method,selected_budget=commonB,comparison='common_budget',
                   is_budgeted=True,convergence_certified=bool(gate),reference_dim=REFERENCE_PZ)
        outputs.append(row);preds[tag]=pp;histories[tag]=inner_histories[method]
    # Posterior-mean ablation first; uncertainty-aware proposed method LAST.
    for mi,method in enumerate(bayesian_methods,1):
        sr=selected.loc[selected.method==method].iloc[0]
        budget=int(sr.budget);dims=validated[(method,budget)]
        for j,c in enumerate(CATS):
            allocrows.append(dict(method=method,predictor=c,dimension=int(validated[(method,commonB)][j]),
               allocated_parameters=int(n[j]*validated[(method,commonB)][j]),common_budget=commonB))
        report('BAYESIAN FINAL',f'{method}; selected B={budget:,} | proposed method finishes last',mi,len(bayesian_methods))
        row,pp,h=fit_final_model('embedding',dims,n,final,heldout,method,run_fingerprint)
        row.update(method=method,selected_budget=budget,comparison='selected_budget',
                   is_budgeted=True,convergence_certified=bool(gate),reference_dim=REFERENCE_PZ)
        outputs.append(row);preds[method]=pp;histories[method]=inner_histories[method]
        if budget!=commonB:
            tag=method+'_common_budget'
            cb_dim=validated[(method,commonB)]
            report('BAYESIAN FINAL',f'Additional common-B fit: {method} B={commonB:,}')
            row_cb,pp_cb,h_cb=fit_final_model('embedding',cb_dim,n,final,heldout,tag,run_fingerprint)
            row_cb.update(method=method,selected_budget=commonB,comparison='common_budget',
                is_budgeted=True,convergence_certified=bool(gate),reference_dim=REFERENCE_PZ)
            outputs.append(row_cb);preds[tag]=pp_cb;histories[tag]=inner_histories[method]

    results=pd.DataFrame(outputs)
    save_table(results,'heldout_method_comparison')
    alloc_table=pd.DataFrame(allocrows)
    save_table(alloc_table,'common_budget_allocations')
    if CFG.bootstrap_reps > 0:
        pair=paired_bootstrap(heldout[2],preds['bayesian_uncertainty'],preds)
        if len(pair):save_table(pair,'paired_auc_differences')
    report('STAGE 6/6 EXPORT','Generating manuscript-ready tables and figures')
    make_figures(results,preds,histories,alloc_table,grid,B,n,eigs,taildf,heldout[2])
    pd.DataFrame({'order_id':heldout_df.order_id.values,'target':heldout[2],
                  **preds}).to_csv(PRED/'heldout_predictions.csv',index=False)
    (OUT/'README_RESULTS.txt').write_text(
       'Manuscript-aligned Bayesian budget allocation: one 70/30 split; 80/20 internal budget split.\n'
       'Execution: non-Bayesian methods fitted and saved before Bayesian MCMC.\n'
       'Run selection: explicit NEW/RESUME; previous study archived before NEW.\n'
       'SIM2-inspired PT: full RNG/state checkpoint, roundtrip labels and cache-consistency audits.\n'
       'All allocation methods evaluated on common data-derived embedding budgets.\n'
       'Budget selection: MIN B with inner-validation binary logloss <= Bmax loss + '+str(CFG.budget_tolerance)+'.\n'
       'MLP from EmbDim.ipynb: hidden=(32,8), ReLU, dropout='+str(CFG.dropout)+', batch_size=32, Adam, 50 epochs.\n'
       'No patience, early stopping, epoch tuning or fixed per-feature dimensions.\n'
       'Reference weighted centering and fixed-map hierarchical Gamma/Beta Bayesian spectral posterior.\n'
       'Posterior uses Bernoulli likelihood, unlike the manuscript Gaussian-response specification.\n'
       'Gaussian MSE theorems cannot be carried over to Bernoulli logloss without additional proof.\n'
       'Convergence gate: '+str(gate)+'; posterior subset size='+str(len(bayesdata[2]))+' of '+str(len(inner[2]))+'.\n'
       'This single split does not estimate across-split uncertainty. Holdout never used to select budget.\n'
       'Paired held-out bootstrap disabled (0 repetitions).\n')
    archive=str(OUT/'olist_manuscript_outputs.zip')
    with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as zf:
        for folder in (TAB,FIG,POST):
            for f in folder.iterdir():
                if CFG.bootstrap_reps == 0 and folder==TAB and f.name=='paired_auc_differences.csv':
                    continue  # Do not bundle a stale bootstrap table from a prior run.
                if f.is_file() and (folder!=POST or f.name in ('posterior_status.json','mcmc_diagnostics_final.csv')):
                    zf.write(f,arcname=str(f.relative_to(OUT)))
        for f in (OUT/'README_RESULTS.txt',OUT/'dataset_description.json',OUT/'configuration.json',PROGRESS_LOG):
            if f.exists():zf.write(f,arcname=f.name)
    report('DONE',f'All outputs saved under {OUT} | ZIP={archive}')
    print(results[['model','auc','logloss','brier','ece','trainable_params']].to_string(index=False))

if __name__=='__main__':
    main()


CUDA GPU not detected; using CPU.
Mounted at /content/drive

========== OLIST EXPERIMENT: NEW OR RESUME ==========
[1] START NEW — archive any prior results and reset training + MCMC
[2] RESUME — reuse finished fits and exact Bayesian PT checkpoint
Experiment directory: /content/drive/MyDrive/JRSSB_Embedding_Allocation/OLIST_BAYESIAN_EJOR_GPU_MANUAL_UPLOAD_V4
Existing study detected; compatible saved work: YES
Choose 1=NEW or 2=RESUME: 2
RESUMING prior study. Existing fits and posterior will not be erased.
Compute device: cpu | posterior LL: NumPy CPU | output: /content/drive/MyDrive/JRSSB_Embedding_Allocation/OLIST_BAYESIAN_EJOR_GPU_MANUAL_UPLOAD_V4 | mode: resume
[2026-11-11 08:28:15] STAGE 1/6 DATA: RESUME mode | request/reuse user Olist files; fixed split seed=42
Reusing the Olist files that YOU uploaded previously, saved at: /content/drive/MyDrive/JRSSB_Embedding_Allocation/OLIST_BAYESIAN_EJOR_GPU_MANUAL_UPLOAD_V4/user_supplied_olist_csvs
All nine user-supplied Olist CSVs verified